# 🧪 PHASE 2 FINAL TEST BENCHMARK: 4 DATASETS (100% TRAIN -> OFFICIAL TEST SET)
* **Ngày cập nhật:** 2026-10-21
* **Tài liệu lý thuyết & Hướng dẫn kỹ thuật:** [`docs/SWIN_SO_KAC_RESEARCH_TRACKER.md`](docs/SWIN_SO_KAC_RESEARCH_TRACKER.md)
* **Notebook gốc đối chứng (Base):** `notebooks/apt/Unified_SO_KAC_3ReLU_4Datasets.ipynb` (ViT-B/16 Baseline)
* **Mục tiêu nghiên cứu:** Phá vỡ trần biểu diễn của vector đơn `[CLS]` ($768$ chiều) trên `ViT-B/16` (`85.8M` params, `17.6 GFLOPs`) bằng cấu trúc phân cấp đa tỷ lệ (Hierarchical Multi-Scale) của **Swin Transformer** (`Swin-T` / `Swin-S`), giảm **$45\% \sim 68\%$ chi phí backbone** trong khi **giữ nguyên 100% ngân sách bộ nhớ Gram Matrix ($121.5\text{ MB}$ RAM)** của `SO_KAC_3ReLU`.
* **GIAI ĐOẠN HIỆN TẠI: PHASE 2 (FINAL TEST BENCHMARK)**: Huấn luyện trên 100% tập Train và đánh giá trực tiếp trên Tập Test Chính Thức (`raw_test`) của bài báo (Zero Test-Leakage). 4 Milestone Models: `Exp0_Baseline` (ViT 17.6G), `Exp1b_SwinS_F1_P0` (Frozen Swin-S 8.7G), `Exp5_SwinS_F3_P5` (Swin-S + MultiStage F3 + ConvPass/RPB), và `Exp6a_SwinS_FlipGram` (Full Proposed Method).
* **Cơ chế chống tràn RAM (57GB Kaggle `/tmp`):** Tích hợp Dual-Mode Caching: FP16 Tensor cấp phát trực tiếp trong RAM (cho ImageNet-R 24k ảnh) + Tự động kích hoạt Memmap FP16 trên `/tmp` NVMe 57GB (cho CIFAR-100 50k ảnh), triệt tiêu 100% rủi ro OOM 30GB CPU RAM!
---
### 🔬 SƠ ĐỒ 3 TRỤC THỬ NGHIỆM ABLATION (10 CẤU HÌNH ĐỘC LẬP)
| Mã Thử Nghiệm | Backbone | Fusion ($d_{\text{in}} + d_{\text{orth}}$) | First-Session PEFT (Stage 3+4) | RAM $G$ | Mục Tiêu & Cơ Chế Toán Học |
| :--- | :--- | :--- | :--- | :---: | :--- |
| **`Exp0_Baseline`** | `ViT-B/16` | `[CLS]` ($768+1536$) | `APTPrompt` (12 Layers) | `121.5 MB` | Chuẩn đối chứng APT gốc (ICCV 2025) trên ViT |
| **`Exp1a_SwinT_F1_P0`** | `Swin-T` | `F1_gap` ($768+1536$) | `P0_Frozen` (Zero PEFT, $E_1=0$) | `121.5 MB` | Đo trần đặc trưng đông băng của Swin-T ($4.5\text{ GFLOPs}$) |
| **`Exp1b_SwinS_F1_P0`** | `Swin-S` | `F1_gap` ($768+1536$) | `P0_Frozen` (Zero PEFT, $E_1=0$) | `121.5 MB` | Đo trần đặc trưng đông băng của Swin-S ($8.7\text{ GFLOPs}$) |
| **`Exp2a_SwinS_F2_P0`** | `Swin-S` | `F2_avgmax` ($768+1536$) | `P0_Frozen` (Zero PEFT, $E_1=0$) | `121.5 MB` | Kết hợp $\text{GAP} + 0.35\cdot\text{GMP}$ bổ sung tiêu điểm nổi bật |
| **`Exp2b_SwinS_F3_P0`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P0_Frozen` (Zero PEFT, $E_1=0$) | `121.5 MB` | Ghép $[	ext{LN}(	ext{GAP}_3), 	ext{LN}(	ext{GAP}_4)]$, $d_{\text{orth}}=1152$ giữ $121.5\text{MB}$ |
| **`Exp3a_SwinS_F3_P1`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P1_SwinAPT` (Window-Positional) | `121.5 MB` | Prompt $P_k, P_v \in \mathbb{R}^{1\times H_a \times 49 \times d_h}$ chống triệt tiêu Softmax |
| **`Exp3b_SwinS_F3_P2`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P2_Adapter` (Parallel Bottleneck) | `121.5 MB` | Bottleneck $r=16, s=0.5, W_{\text{up}}=0$ song song MLP |
| **`Exp3c_SwinS_F3_P3`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P3_SSF` (Scale & Shift) | `121.5 MB` | Điều biến đặc trưng $\gamma \odot x + \beta$ sau norm1, norm2 |
| **`Exp4_SwinS_F3_P4`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P4_convpass` (2D Spatial DW-Conv 3x3) | `121.5 MB` | ConvPass $3\times3$ Depthwise Conv xuyên biên cửa sổ $7\times7$, nối thông đặc trưng không gian |
| **`Exp5_SwinS_F3_P5`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P5_hybrid_conv_rpb` (ConvPass + Delta RPB) | `121.5 MB` | Mốc cơ sở: ConvPass 2D + Delta RPB ở Stage 3-4 |
| **`Exp6a_SwinS_FlipGram`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P5_hybrid_conv_rpb` + Flip/Power | `121.5 MB` | [+Bước 1] 2-View Flip Feature Averaging & Tukey Power-Norm $eta=0.85$ (Tái sử dụng PEFT Exp5, 0s extra train) |
| **`Exp6b_SwinS_FullStageDA`** | `Swin-S` | `F3_multistage` ($1152+1152$) | `P6_fullstage_da` (Early SSF + ConvPass) | `121.5 MB` | [+Bước 2] Mở khóa Early-Stage Domain SSF ($\gamma, eta$) ở patch_embed, Stage 1 & 2 (~3k params) |
| **`Exp6c_SwinS_FgAttnPool`** | `Swin-S` | `F4_fg_multistage` ($1152+1152$) | `P6_fullstage_da` + FgAttnPool | `121.5 MB` | [+Bước 3] Foreground Salience Attention Pooling thay GAP mù, lọc nhiễu nền |
| **`Exp6d_SwinS_SupConMix`** | `Swin-S` | `F4_fg_multistage` ($1152+1152$) | `P6_fullstage_da` + SupCon/Mixup | `121.5 MB` | [+Bước 4] Supervised Contrastive Loss ($	au=0.10$) + Feature Mixup ở Task 1 chống co cụm FDR |
---
### ⚠️ BẢN VÁ TOÁN HỌC & ĐỒNG BỘ ENGINE ĐÃ ĐƯỢC ÁP DỤNG 100%:
1. **Đồng bộ Engine Nghiệm Đóng Chuẩn:** Tích hợp `find_optimal_lambda_fast()` chia 80/20 nội bộ trên Task 1 để chọn $\lambda^* \in [10^2, 10^4]$, loại bỏ hoàn toàn bẫy overfit của $\lambda = 10^{-8}$.
2. **Cân Bằng Phân Phối Chuẩn Xác:** Class-Prior Balancing (`cb_gamma=0.32`) áp dụng trực tiếp lên ma trận mục tiêu $Q_h$ khi giải Ridge, không trừ trực tiếp vào logits gây sụp đổ argmax.
3. **Ánh Xạ Nhãn RemappedSubset Chuẩn:** Gán lại nhãn $y \in [0, C-1]$ theo đúng thứ tự 10 tasks, bảo đảm Task-Aware Accuracy (TA) tính toán chính xác $100\%$.
4. **Bảo toàn Ngân sách Bộ nhớ Gram Matrix:** Khi dùng $1152$ chiều Multi-Stage ($F3$), khóa cứng $\text{head\_dim} = 2304$ qua $d_{\text{skip}}=768, d_{\text{orth}}=1536$, bảo đảm $\text{RAM}(G) = 6 \times 2304^2 \times 4 = 121.50\text{ MB}$ chính xác tuyệt đối.
5. **Nâng Cấp PEFT Toàn Diện & Chẩn Đoán Hình Học (Swin-Specific PEFT & Geometric Metrics):**
   - **`P4_convpass`:** Adapter không gian 2D với $3\times3$ Depthwise Conv phá vỡ rào cản chia cắt của cửa sổ $7\times7$.
   - **`P5_hybrid_conv_rpb`:** Kết hợp ConvPass 2D và hiệu chỉnh bảng độ lệch vị trí tương đối ($\Delta B$).
   - **EMA Anchoring ($\alpha_{\text{ema}}=0.7$):** Co cụm $30\%$ trọng số về pre-trained ở cuối Task 1 chống quá khớp phân phối sớm.
   - **CosineHead Dim-Scaling ($\sqrt{768/D}$):** Giữ nguyên 100% độ lớn logit của ViT gốc ($768\text{d}$), tự động co tỷ lệ cho Swin F3 ($1152\text{d}$).
   - **Fisher Discriminant Ratio ($\text{FDR}_{T1}, \text{FDR}_{\text{unseen}}$):** Đo đạc trực tiếp độ phân tách cụm lớp Task 1 vs 9 tasks chưa học.
---
### 🛡️ QUY TRÌNH 2 GIAI ĐOẠN CHỐNG RÒ RỈ TẬP TEST (ZERO TEST-LEAKAGE):
* **Giai đoạn 1 (`phase="val"`):** 80% Train / 20% Val qua 3 seed `0, 1, 2` (5-Fold Disjoint). Tập Test được giữ nguyên vẹn.
* **Giai đoạn 2 (`phase="test"`):** Final Test CHỈ trên các mô hình Swin chốt ưu tú và Paper Baselines (`Linear Ridge`, `RanPAC 10k`, `RanPAC Iso-RAM`) trên 100% Train gộp.


In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, THƯ VIỆN & THIẾT BỊ GPU
# ==============================================================================
import os
import sys
import math
import time
import json
import random
import pickle
import shutil
import subprocess
import warnings
from collections import OrderedDict
warnings.filterwarnings("ignore")
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_KAGGLE = os.path.exists("/kaggle")
print(f"[*] Môi trường: {'Kaggle GPU' if IS_KAGGLE else 'Local Machine'} | Thiết bị: {DEVICE} | PyTorch: {torch.__version__} | Timm: {timm.__version__}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# ==============================================================================
# CELL 3: CẤU HÌNH THỰC NGHIỆM SWIN TRANSFORMER SO-KAC-RIDGE
# ==============================================================================
DATASET_SPECS = OrderedDict([
    ("cars196", dict(
        dataset_name="Cars196", n_classes=196, task_sizes=[20]*9 + [16], aug_type="imagenet",
        # [CẤU HÌNH VÀNG J2 CHO SWIN FIRST SESSION & ABLATION - HIGH CAPACITY REGIME]:
        peft_lr=0.003, head_lr_ratio=2.5, batch_size=32, epochs=40, warmup_epochs=2,
        rank=64, scale=0.8, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.90,
        cb_gamma=0.18, wa_beta=0.30,
        eval_batch_size=128,
        vit_prompt_lr=0.005, vit_epochs=50, vit_bs=128, vit_ema=0.70,
        kaggle_candidates=[
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data",
            "./data/car_data/car_data",
        ]
    )),
    ("imagenet_r", dict(
        dataset_name="ImageNet_R", n_classes=200, task_sizes=[20]*10, aug_type="imnet_r",
        # [CẤU HÌNH VÀNG INR_J CHO SWIN FIRST SESSION & ABLATION - HIGH CAPACITY REGIME]:
        peft_lr=0.004, head_lr_ratio=2.5, batch_size=32, epochs=30, warmup_epochs=2,
        rank=64, scale=0.6, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.85,
        cb_gamma=0.18, wa_beta=0.30,
        eval_batch_size=128,
        vit_prompt_lr=0.003, vit_epochs=30, vit_bs=128, vit_ema=0.80,
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r",
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r/imagenet-r",
            "/kaggle/input/imagenet-r",
            "/kaggle/input/imagenet-r/imagenet-r",
            "./data/imagenet-r",
        ]
    )),
    ("cub200", dict(
        dataset_name="Cub200", n_classes=200, task_sizes=[20]*10, aug_type="cub",
        # [CẤU HÌNH VÀNG CUB_J2 CHO SWIN FIRST SESSION & ABLATION - GENTLE REFINEMENT REGIME]:
        peft_lr=0.002, head_lr_ratio=2.0, batch_size=64, epochs=25, warmup_epochs=2,
        rank=32, scale=0.3, adapter_dropout=0.1, weight_decay=0.001, use_cosine_sched=True, ema_coeff=0.85,
        cb_gamma=0.0, wa_beta=0.0,
        eval_batch_size=128,
        vit_prompt_lr=0.02, vit_epochs=20, vit_bs=128, vit_ema=0.70,
        kaggle_candidates=[
            "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
            "/kaggle/input/cub-200-2011-by-classes-folder",
            "/kaggle/input/cyizhuo-cub-200-2011-by-classes-folder",
            "/kaggle/input/cub-200-2011",
            "/kaggle/input/cub2002011",
            "./data/cub-200-2011-by-classes-folder",
            "./data/cub200",
        ]
    )),
    ("cifar100", dict(
        dataset_name="Cifar100", n_classes=100, task_sizes=[10]*10, aug_type="cifar",
        # [CẤU HÌNH VÀNG CIF_J CHO SWIN FIRST SESSION & ABLATION - BALANCED FAST-CONV REGIME]:
        peft_lr=0.004, head_lr_ratio=2.0, batch_size=64, epochs=20, warmup_epochs=2,
        rank=32, scale=0.3, adapter_dropout=0.1, weight_decay=0.001, use_cosine_sched=True, ema_coeff=0.85,
        cb_gamma=0.0, wa_beta=0.0,
        eval_batch_size=128,
        vit_prompt_lr=0.004, vit_epochs=20, vit_bs=128, vit_ema=0.70,
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
            "/kaggle/input/cifar-100-python/cifar-100-python",
            "/kaggle/input/cifar-100-python",
            "./data/cifar-100-python",
        ]
    )),
])
# 1. Danh sách Datasets đang kích hoạt:
AVAILABLE_DATASETS = ["cars196", "cub200", "imagenet_r", "cifar100"]
# 2. Bộ siêu tham số điều khiển thực nghiệm:
SEEDS_TO_RUN      = [0, 1, 2]
RUN_VAL_ABLATION   = False  # Đã hoàn thành 100% trên cả 4 datasets!
RUN_FINAL_TEST     = True   # Bật Phase 2: Huấn luyện trên 100% Train -> Đánh giá trên Official Test Set!
USE_AMP           = True
NUM_WORKERS       = 2 if IS_KAGGLE else 0
MASTER_OUT_DIR    = "/kaggle/working/outputs/swin_so_kac_master" if IS_KAGGLE else "./results/apt/swin_so_kac_master"
os.makedirs(MASTER_OUT_DIR, exist_ok=True)
RIDGE_LAMBDAS = [v for p in range(-8, 9) for v in (10.0**p, 3.0 * (10.0**p))]
# 3. Danh mục 8 Cấu hình Swin Ablation Độc Lập (Table 2 - SWIN_SO_KAC_RESEARCH_TRACKER.md):
SWIN_ABLATION_EXPERIMENTS = OrderedDict([
    ("Exp0_Baseline", dict(
        backbone="vit_base", fusion="CLS", peft="vit_apt",
        desc="ViT-B/16 Baseline (17.6 GFLOPs, 12L APT Prompt, CLS 768d)"
    )),
    ("Exp1a_SwinT_F1_P0", dict(
        backbone="swin_tiny", fusion="F1_gap", peft="P0_frozen",
        desc="Swin-T Frozen GAP (4.5 GFLOPs, -74% FLOPs, Stage 4 GAP 768d)"
    )),
    ("Exp1b_SwinS_F1_P0", dict(
        backbone="swin_small", fusion="F1_gap", peft="P0_frozen",
        desc="Swin-S Frozen GAP (8.7 GFLOPs, -51% FLOPs, Stage 4 GAP 768d)"
    )),
    ("Exp2a_SwinS_F2_P0", dict(
        backbone="swin_small", fusion="F2_avgmax", peft="P0_frozen",
        desc="Swin-S Frozen AvgMax (8.7 GFLOPs, GAP + 0.35 GMP, 768d)"
    )),
    ("Exp2b_SwinS_F3_P0", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P0_frozen",
        desc="Swin-S Multi-Stage (8.7 GFLOPs, Stage 3+4 Concat LN, 1152d)"
    )),
    ("Exp3a_SwinS_F3_P1", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P1_swin_apt",
        desc="Swin-S Multi-Stage + P1 Window-Positional APT (49 tokens/window)"
    )),
    ("Exp3b_SwinS_F3_P2", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P2_adapter",
        desc="Swin-S Multi-Stage + P2 Parallel Bottleneck Adapter (r=16, s=0.5)"
    )),
    ("Exp3c_SwinS_F3_P3", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P3_ssf",
        desc="Swin-S Multi-Stage + P3 Scale & Shift Feature Modulation (SSF)"
    )),
    ("Exp4_SwinS_F3_P4", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P4_convpass",
        desc="Swin-S Multi-Stage + P4 ConvPass 2D Adapter (r=16, 3x3 DW-Conv, Cross-Window)"
    )),
    ("Exp5_SwinS_F3_P5", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        flip_gram=False, supcon_mix=False,
        desc="Swin-S Multi-Stage + P5 Hybrid ConvPass 2D + Delta RelPosBias Tuning"
    )),
    # --- CHUỖI BÓC TÁCH STEPWISE FEW-SHOT & DOMAIN ADAPTATION (EXP6a -> EXP6d) ---
    ("Exp6a_SwinS_FlipGram", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        flip_gram=True, supcon_mix=False,
        desc="[+Bước 1] Exp5 + Few-Shot 2-View Flip Feature & Power-Norm Gram"
    )),
    ("Exp6b_SwinS_FullStageDA", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P6_fullstage_da",
        flip_gram=True, supcon_mix=False,
        desc="[+Bước 2] Exp6a + Mở khóa Early-Stage Domain SSF (Stage 1 & Stage 2)"
    )),
    ("Exp6c_SwinS_FgAttnPool", dict(
        backbone="swin_small", fusion="F4_fg_multistage", peft="P6_fullstage_da",
        flip_gram=True, supcon_mix=False,
        desc="[+Bước 3] Exp6b + Foreground Salience Attention Pooling (Loại nhiễu nền)"
    )),
    ("Exp6d_SwinS_SupConMix", dict(
        backbone="swin_small", fusion="F4_fg_multistage", peft="P6_fullstage_da",
        flip_gram=True, supcon_mix=True,
        desc="[+Bước 4] Exp6c + Supervised Contrastive & Manifold Mixup ở Task 1"
    )),
    # ==============================================================================
    # CẤU HÌNH KIỂM CHỨNG TƯƠNG TÁC CHÉO (2D JOINT GRID SEARCH TRÊN SO-KAC-RIDGE)
    # ==============================================================================
    ("J0_Anchor_BS128", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=128, peft_lr=0.005, head_lr_ratio=1.0, epochs=50, warmup_epochs=0,
        scale=0.5, rank=16, adapter_dropout=0.0, weight_decay=0.0, use_cosine_sched=False, ema_coeff=0.70,
        desc="[Mốc Exp5 cũ] BS=128, lr=5e-3, r=16, s=0.5, ep=50, ema=0.7 (72.44%)"
    )),
    ("J1_BS32_SafeTuned", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.002, head_lr_ratio=2.0, epochs=35, warmup_epochs=2,
        scale=0.5, rank=32, adapter_dropout=0.1, weight_decay=1e-4, use_cosine_sched=True, ema_coeff=0.85,
        desc="[Ứng viên 1] BS=32, lr=2e-3, ratio=2, r=32, s=0.5, ep=35, warm=2, ema=0.85"
    )),
    ("J2_BS32_HighRank64", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.003, head_lr_ratio=2.5, epochs=40, warmup_epochs=2,
        scale=0.8, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.90,
        desc="[Ứng viên 2] BS=32, lr=3e-3, ratio=2.5, r=64, s=0.8, wd=0.01, drop=0.1, ema=0.9"
    )),
    ("J3_BS64_Balanced", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=64, peft_lr=0.005, head_lr_ratio=2.5, epochs=40, warmup_epochs=2,
        scale=0.8, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.90,
        desc="[Ứng viên 3] BS=64, lr=5e-3, ratio=2.5, r=64, s=0.8, wd=0.01, drop=0.1, ema=0.9"
    )),
    ("J4_JSON_RawBest", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.020, head_lr_ratio=5.0, epochs=50, warmup_epochs=2,
        scale=1.0, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=False, ema_coeff=1.00,
        desc="[Raw JSON Best] BS=32, lr=2e-2, ratio=5, r=64, s=1.0, ep=50, ema=1.0"
    )),
])
# CHUỖI STEPWISE ABLATION 7 BẬC CHỨNG MINH CÔNG NĂNG TỪNG KHỐI TRONG SWIN:
ACTIVE_ABLATION_EXPERIMENTS = [
    "Exp0_Baseline",           # 1. [ViT Baseline] ViT-B/16 + APT (17.6 GFLOPs) -> Mốc đối chứng kiến trúc cũ
    "Exp1b_SwinS_F1_P0",       # 2. [Frozen Swin Baseline] Swin-S Frozen Stage 4 GAP (8.7 GFLOPs, 0s train)
    "Exp5_SwinS_F3_P5",        # 3. [Our Swin + PEFT] Swin-S + Multi-Stage F3 + Hybrid ConvPass-RPB
    "Exp6a_SwinS_FlipGram",    # 4. [Our Full Proposed Method] Exp5 + 2-View Flip & Power-Norm Gram (0s extra train!)
]

ACTIVE_HEADS = [
    "B0_linear_ridge",         # [Head Baseline 1] Linear Ridge (5.1 MB RAM)
    "SO_KAC_3ReLU",            # [Proposed Head]   Calibrated Sobolev KAC v5 Champion (121.5 MB RAM)
    "B3_ranpac_ram5640",       # [Iso-RAM Baseline] RanPAC M=5640 (121.3 MB RAM - Đối chứng công bằng cùng bộ nhớ)
    "B2_ranpac_10k",           # [Paper Baseline]   RanPAC M=10000 (381.5 MB RAM)
]
print(f"[*] Đã nạp cấu hình Swin Phase 2 Final Test: Datasets: {AVAILABLE_DATASETS} | Thí nghiệm: {ACTIVE_ABLATION_EXPERIMENTS} | Heads: {ACTIVE_HEADS}")

In [ ]:
# ==============================================================================

# CELL 4: NẠP DỮ LIỆU, CHIA 5-FOLD DISJOINT & ZERO-DISK I/O RAM CACHING

# ==============================================================================

class CIFAR100PickleDataset(Dataset):

    def __init__(self, root, train=True):

        split_file = "train" if train else "test"

        with open(os.path.join(root, split_file), "rb") as f:

            entry = pickle.load(f, encoding="latin1")

        self.data = entry["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        self.targets = [int(y) for y in entry["fine_labels"]]

    def __len__(self):

        return len(self.targets)

    def __getitem__(self, idx):

        return Image.fromarray(self.data[idx]), self.targets[idx]



class CachedTrainSubset(Dataset):

        # Zero-Disk I/O Train Subset: Tự động giải nén và giữ nguyên PIL Image trong RAM (img.load()).
    # Chỉ thực thi Random Augment khi DataLoader lấy mẫu, triệt tiêu 100% độ trễ đọc đĩa lặp lại qua 40 epochs!

    def __init__(self, raw_samples, indices, remap, tf, is_cifar=False):

        self.tf = tf

        self.is_cifar = is_cifar

        self.samples = []

        for idx in indices:

            if is_cifar:

                data_np, targets = raw_samples

                img = Image.fromarray(data_np[idx])

                y = remap[int(targets[idx])]

            else:

                path, orig_y = raw_samples[idx]

                img = Image.open(path).convert("RGB")

                img.load()  # Nạp sẵn toàn bộ byte ảnh vào RAM!

                y = remap[orig_y]

            self.samples.append((img, y))

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, i):

        img, y = self.samples[i]

        return self.tf(img), y



class PreloadedTensorDataset(Dataset):
    """
    Dataset chứa trực tiếp Tensor (C, H, W) lưu ở FP16 trong RAM hoặc mmap trên /tmp.
    Tự động chuyển đổi sang float32 trong __getitem__ khi DataLoader nạp batch vào GPU.
    """
    def __init__(self, tensors, labels, is_mmap=False, mmap_path=None):
        self.tensors = tensors
        self.labels = labels
        self.is_mmap = is_mmap
        self.mmap_path = mmap_path

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if self.is_mmap:
            t = torch.from_numpy(self.tensors[idx].copy()).float()
        else:
            t = self.tensors[idx].float()
        return t, self.labels[idx]

    def cleanup(self):
        if self.is_mmap and self.mmap_path and os.path.exists(self.mmap_path):
            try:
                del self.tensors
                os.remove(self.mmap_path)
            except Exception:
                pass

    def __del__(self):
        self.cleanup()

def precompute_eval_tensors(raw_samples, indices, remap, eval_tf, is_cifar=False, desc="Eval", force_mmap=False):
    """
    Tiền tính toán toàn bộ ảnh eval (Resize + CenterCrop) thành Tensor FP16 (N, 3, 224, 224).
    - Mặc định (N < 25,000 như Cars/CUB/ImageNet-R): Cấp phát trước FP16 trong RAM (giảm 50% RAM, siêu tốc Zero-Disk I/O).
    - Tự động (N >= 25,000 như CIFAR-100 50,000 ảnh): Ghi trực tiếp ra file np.memmap FP16 trên /tmp
      (tận dụng trọn vẹn 57GB ổ tạm NVMe của Kaggle), đưa RAM tiêu thụ về ~0 MB, triệt tiêu 100% rủi ro OOM!
    """
    N = len(indices)
    t0 = time.time()
    tmp_dir = "/tmp" if os.path.exists("/tmp") else tempfile.gettempdir()
    should_mmap = force_mmap or (N >= 25000 and os.path.exists(tmp_dir))

    if should_mmap:
        mmap_file = os.path.join(tmp_dir, f"eval_cache_{int(time.time()*1000)}_{N}.mmap")
        print(f"[*] Đang tiền xử lý & ghi {N} ảnh {desc} vào ổ đệm siêu tốc /tmp (mmap FP16, RAM=0MB)...")
        mmap_arr = np.memmap(mmap_file, dtype=np.float16, mode='w+', shape=(N, 3, 224, 224))
        l_stack = torch.empty(N, dtype=torch.long)
        for i, idx in enumerate(indices):
            if is_cifar:
                data_np, targets = raw_samples
                img = Image.fromarray(data_np[idx])
                y = remap[int(targets[idx])]
            else:
                path, orig_y = raw_samples[idx]
                with Image.open(path) as im:
                    img = im.convert("RGB")
                y = remap[orig_y]
            mmap_arr[i] = eval_tf(img).half().numpy()
            l_stack[i] = y
        mmap_arr.flush()
        read_mmap = np.memmap(mmap_file, dtype=np.float16, mode='r', shape=(N, 3, 224, 224))
        size_mb = os.path.getsize(mmap_file) / (1024 * 1024)
        print(f"    -> Hoàn tất nạp {N} ảnh vào /tmp mmap trong {time.time() - t0:.1f}s (Dung lượng đĩa: {size_mb:.1f} MB | RAM: ~0 MB)")
        return PreloadedTensorDataset(read_mmap, l_stack, is_mmap=True, mmap_path=mmap_file)
    else:
        print(f"[*] Đang tiền xử lý & nạp {N} ảnh {desc} vào RAM (Zero-Disk I/O FP16)...")
        t_stack = torch.empty((N, 3, 224, 224), dtype=torch.float16)
        l_stack = torch.empty(N, dtype=torch.long)
        for i, idx in enumerate(indices):
            if is_cifar:
                data_np, targets = raw_samples
                img = Image.fromarray(data_np[idx])
                y = remap[int(targets[idx])]
            else:
                path, orig_y = raw_samples[idx]
                with Image.open(path) as im:
                    img = im.convert("RGB")
                y = remap[orig_y]
            t_stack[i].copy_(eval_tf(img).half())
            l_stack[i] = y
        ram_mb = (t_stack.element_size() * t_stack.nelement()) / (1024 * 1024)
        print(f"    -> Hoàn tất nạp {N} tensor vào RAM trong {time.time() - t0:.1f}s (Dung lượng RAM: {ram_mb:.1f} MB)")
        return PreloadedTensorDataset(t_stack, l_stack, is_mmap=False)



class RemappedSubset(Dataset):

        # Bọc Dataset gốc theo danh sách chỉ số indices, ánh xạ lại nhãn sang [0..C-1] theo remap và áp dụng tf.

    def __init__(self, raw_samples, indices, remap, tf, is_cifar=False):

        self.raw_samples = raw_samples

        self.indices = list(indices)

        self.remap = remap

        self.tf = tf

        self.is_cifar = is_cifar

        if is_cifar:

            data_np, targets = raw_samples

            self.targets = [int(remap[int(targets[i])]) for i in self.indices]

        else:

            self.targets = [int(remap[int(raw_samples[i][1])]) for i in self.indices]

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, i):

        raw_idx = self.indices[i]

        if self.is_cifar:

            data_np, _ = self.raw_samples

            img = Image.fromarray(data_np[raw_idx])

        else:

            path, _ = self.raw_samples[raw_idx]

            with Image.open(path) as im:

                img = im.convert("RGB")

        return self.tf(img), self.targets[i]



def build_transforms(aug_type):

    if aug_type in ["cifar", "cifar10"]:

        train_tf = T.Compose([

            T.AutoAugment(T.AutoAugmentPolicy.CIFAR10, T.InterpolationMode.BILINEAR),

            T.RandomResizedCrop(224),

            T.RandomHorizontalFlip(),

            T.ToTensor(),

        ])

    elif aug_type == "imnet_r":

        train_tf = T.Compose([

            T.AutoAugment(T.AutoAugmentPolicy.IMAGENET, T.InterpolationMode.BILINEAR),

            T.Resize(256),

            T.RandomResizedCrop(224),

            T.RandomHorizontalFlip(),

            T.ToTensor(),

        ])

    else:  # "imagenet", "cub" (CUB-200 & Stanford Cars-196)

        train_tf = T.Compose([

            T.AutoAugment(T.AutoAugmentPolicy.IMAGENET, T.InterpolationMode.BILINEAR),

            T.RandomResizedCrop(224),

            T.RandomHorizontalFlip(),

            T.ToTensor(),

        ])

    eval_tf = T.Compose([

        T.Resize(256),

        T.CenterCrop(224),

        T.ToTensor(),

    ])

    return train_tf, eval_tf



def locate_dataset_root(candidates):

    for path in candidates:

        if os.path.exists(path):

            return path

    raise FileNotFoundError(f"Không tìm thấy thư mục dataset. Đã thử: {candidates}")



def resolve_dataset_dirs(root):

    # 1. Kiểm tra trực tiếp tại root có sẵn cặp train & test:

    tr = os.path.join(root, "train")

    te = os.path.join(root, "test")

    if os.path.isdir(tr) and os.path.isdir(te):

        return tr, te



    # 2. Quét đệ quy os.walk tìm cặp thư mục train và test (Ưu tiên số 1):

    for dirpath, dirnames, _ in os.walk(root):

        if "train" in dirnames and "test" in dirnames:

            tr = os.path.join(dirpath, "train")

            te = os.path.join(dirpath, "test")

            subdirs = [d for d in os.listdir(tr) if os.path.isdir(os.path.join(tr, d))]

            if len(subdirs) >= 10:

                return tr, te



    # 3. Kiểm tra các subpath thông dụng cho cặp train/test:

    common_subpaths = [

        ("car_data", "car_data"),

        ("car_data",),

        ("cars_train",),

        ("CUB_200_2011", "images"),

        ("images",),

    ]

    for sub in common_subpaths:

        cand_root = os.path.join(root, *sub)

        tr = os.path.join(cand_root, "train")

        te = os.path.join(cand_root, "test")

        if os.path.isdir(tr) and os.path.isdir(te):

            return tr, te



    # 4. Chỉ rơi vào nhánh này nếu dataset hoàn toàn không có train/test (như ImageNet-R gộp):

    for sub in common_subpaths:

        cand_root = os.path.join(root, *sub)

        if os.path.isdir(cand_root):

            subdirs = [d for d in os.listdir(cand_root) if os.path.isdir(os.path.join(cand_root, d))]

            if len(subdirs) >= 10 and "train" not in subdirs:

                return cand_root, None



    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]

    if len(subdirs) >= 10:

        return root, None

    return None, None



def load_raw_dataset_samples(d_key, spec):

    root = locate_dataset_root(spec["kaggle_candidates"])

    train_samples, test_samples = [], []

    if d_key in ["cub200", "cars196", "imagenet_r"]:

        tr_dir, te_dir = resolve_dataset_dirs(root)

        if tr_dir is not None and te_dir is not None:

            classes = sorted([d for d in os.listdir(tr_dir) if os.path.isdir(os.path.join(tr_dir, d))])

            c2i = {c: i for i, c in enumerate(classes)}

            for sub_dir, target in [(tr_dir, train_samples), (te_dir, test_samples)]:

                for c in classes:

                    cd = os.path.join(sub_dir, c)

                    if not os.path.isdir(cd):

                        continue

                    for fname in sorted(os.listdir(cd)):

                        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):

                            target.append((os.path.join(cd, fname), c2i[c]))

        elif tr_dir is not None:

            classes = sorted([d for d in os.listdir(tr_dir) if os.path.isdir(os.path.join(tr_dir, d))])

            c2i = {c: i for i, c in enumerate(classes)}

            for c in classes:

                cd = os.path.join(tr_dir, c)

                fnames = sorted([f for f in os.listdir(cd) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

                s_list = [(os.path.join(cd, fn), c2i[c]) for fn in fnames]

                n_tr = int(0.8 * len(s_list)) if d_key != "cub200" else int(0.5 * len(s_list))

                train_samples.extend(s_list[:n_tr])

                test_samples.extend(s_list[n_tr:])

        else:

            raise ValueError(f"Không thể định vị thư mục train/test hoặc các lớp trong: {root}")

    elif d_key == "cifar100":

        def unpickle(fp):

            with open(fp, 'rb') as f:

                return pickle.load(f, encoding='bytes')

        tr_b = unpickle(os.path.join(root, "train"))

        te_b = unpickle(os.path.join(root, "test"))

        tr_data = tr_b[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        te_data = te_b[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        tr_labels = [int(y) for y in tr_b[b'fine_labels']]

        te_labels = [int(y) for y in te_b[b'fine_labels']]

        return (tr_data, tr_labels), (te_data, te_labels)

    print(f"[*] Đã nạp dataset {d_key}: {len(train_samples)} mẫu train, {len(test_samples)} mẫu test từ {root}")

    if len(train_samples) == 0:

        raise ValueError(f"LỖI: 0 mẫu ảnh được nạp cho {d_key} từ đường dẫn {root}!")

    return train_samples, test_samples



def setup_cil_seed(raw_train, seed, num_classes, task_sizes, n_splits=5, base_seed=1993, is_cifar=False):

    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"

    rng_task = np.random.RandomState(seed)

    class_order = rng_task.permutation(num_classes).tolist()

    remap = {orig_c: new_c for new_c, orig_c in enumerate(class_order)}

    bounds, cur = [], 0

    for sz in task_sizes:

        bounds.append((cur, cur + sz))

        cur += sz

    if is_cifar:

        _, targets = raw_train

        targets = np.asarray(targets)

    else:

        targets = np.asarray([y for _, y in raw_train])

    tr_idx, va_idx = [], []

    rng_split = np.random.RandomState(base_seed)

    for c in range(num_classes):

        c_indices = np.where(targets == c)[0]

        perm = rng_split.permutation(c_indices)

        folds = np.array_split(perm, n_splits)

        va_idx.extend(folds[seed])

        for k in range(n_splits):

            if k != seed:

                tr_idx.extend(folds[k])

    return class_order, remap, bounds, np.array(tr_idx), np.array(va_idx)


In [ ]:
# ==============================================================================

# CELL 5: BACKBONE ARCHITECTURES (VIT-B/16 & SWIN TRANSFORMER) + PEFT MODULES

# ==============================================================================

SWIN_ARCH_MAP = {

    "swin_tiny":  "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",

    "swin_small": "swin_small_patch4_window7_224.ms_in22k_ft_in1k",

}

# --- 1. VIT BACKBONE & ADDITIVE PROMPT (CHO EXP0_BASELINE) ---

class PatchEmbed(nn.Module):

    def __init__(self, dim=768, patch=16):

        super().__init__()

        self.proj = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)

    def forward(self, x):

        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):

    def __init__(self, dim=768, heads=12):

        super().__init__()

        self.heads, self.hd = heads, dim // heads

        self.qkv = nn.Linear(dim, dim * 3, bias=True)

        self.proj = nn.Linear(dim, dim)

    def forward(self, x, prompt=None):

        B, N, C = x.shape

        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, self.hd).permute(2, 0, 3, 1, 4).unbind(0)

        if prompt is not None:

            pk, pv = prompt

            k = torch.cat([k[:, :, :1] + pk.to(k.dtype), k[:, :, 1:]], dim=2)

            v = torch.cat([v[:, :, :1] + pv.to(v.dtype), v[:, :, 1:]], dim=2)

        x = F.scaled_dot_product_attention(q, k, v)

        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Mlp(nn.Module):

    def __init__(self, dim=768, hidden=3072):

        super().__init__()

        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)

    def forward(self, x):

        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):

    def __init__(self, dim=768, heads=12):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim, eps=1e-6)

        self.attn = Attention(dim, heads)

        self.norm2 = nn.LayerNorm(dim, eps=1e-6)

        self.mlp = Mlp(dim, dim * 4)

    def forward(self, x, prompt=None):

        x = x + self.attn(self.norm1(x), prompt)

        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):

    def __init__(self, dim=768, depth=12, heads=12):

        super().__init__()

        self.patch_embed = PatchEmbed(dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))

        self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))

        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])

        self.norm = nn.LayerNorm(dim, eps=1e-6)

        self.out_features = dim

    def forward(self, x, prompt_fn=None):

        B = x.size(0)

        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], dim=1) + self.pos_embed

        for l, blk in enumerate(self.blocks):

            x = blk(x, prompt_fn(l) if prompt_fn else None)

        return self.norm(x)[:, 0]

def load_pretrained_vit(name="vit_base_patch16_224.augreg_in21k_ft_in1k"):

    try:

        ref = timm.create_model(name, pretrained=True, num_classes=0)

    except Exception:

        ref = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)

    sd = ref.state_dict()

    vit = ViT(768, 12, 12)

    with torch.no_grad():

        vit.patch_embed.proj.weight.copy_(sd["patch_embed.proj.weight"])

        vit.patch_embed.proj.bias.copy_(sd["patch_embed.proj.bias"])

        vit.cls_token.copy_(sd["cls_token"])

        vit.pos_embed.copy_(sd["pos_embed"])

        vit.norm.weight.copy_(sd["norm.weight"])

        vit.norm.bias.copy_(sd["norm.bias"])

        for i in range(12):

            vit.blocks[i].norm1.weight.copy_(sd[f"blocks.{i}.norm1.weight"])

            vit.blocks[i].norm1.bias.copy_(sd[f"blocks.{i}.norm1.bias"])

            vit.blocks[i].attn.qkv.weight.copy_(sd[f"blocks.{i}.attn.qkv.weight"])

            vit.blocks[i].attn.qkv.bias.copy_(sd[f"blocks.{i}.attn.qkv.bias"])

            vit.blocks[i].attn.proj.weight.copy_(sd[f"blocks.{i}.attn.proj.weight"])

            vit.blocks[i].attn.proj.bias.copy_(sd[f"blocks.{i}.attn.proj.bias"])

            vit.blocks[i].norm2.weight.copy_(sd[f"blocks.{i}.norm2.weight"])

            vit.blocks[i].norm2.bias.copy_(sd[f"blocks.{i}.norm2.bias"])

            vit.blocks[i].mlp.fc1.weight.copy_(sd[f"blocks.{i}.mlp.fc1.weight"])

            vit.blocks[i].mlp.fc1.bias.copy_(sd[f"blocks.{i}.mlp.fc1.bias"])

            vit.blocks[i].mlp.fc2.weight.copy_(sd[f"blocks.{i}.mlp.fc2.weight"])

            vit.blocks[i].mlp.fc2.bias.copy_(sd[f"blocks.{i}.mlp.fc2.bias"])

    for p in vit.parameters():

        p.requires_grad = False

    return vit.eval()

class APTPrompt(nn.Module):

    def __init__(self, depth=12, dim=768, heads=12, ema_coeff=0.7):

        super().__init__()

        self.depth, self.heads, self.hd = depth, heads, dim // heads

        self.ema_coeff = ema_coeff

        self.prompt_tokens = nn.Parameter(torch.zeros(depth * 2, dim))

        nn.init.trunc_normal_(self.prompt_tokens, std=0.02)

        self.init_state = {k: v.detach().clone() for k, v in self.state_dict().items()}

    def get(self, l):

        P = self.prompt_tokens

        return P[2 * l].view(self.heads, 1, self.hd), P[2 * l + 1].view(self.heads, 1, self.hd)

    def apply_ema_anchoring(self, alpha=0.7):

        with torch.no_grad():

            for k, v in self.named_parameters():

                if k in self.init_state:

                    v.data.copy_(alpha * v.data + (1.0 - alpha) * self.init_state[k].to(v.device))

    def grad_norm(self):

        if self.prompt_tokens.grad is not None:

            return float(self.prompt_tokens.grad.detach().norm(2).item())

        return 0.0

# --- 2. SWIN TRANSFORMER BACKBONE & PEFT MODULES ---

class ConvPass2DAdapter(nn.Module):

    def __init__(self, dim, rank=64, dropout=0.1):

        super().__init__()

        self.dim = dim

        self.rank = rank

        self.down = nn.Linear(dim, rank, bias=True)

        self.act = nn.GELU()

        self.drop = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        self.dwconv = nn.Conv2d(rank, rank, kernel_size=3, stride=1, padding=1, groups=rank, bias=True)

        self.up = nn.Linear(rank, dim, bias=True)

        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))

        nn.init.zeros_(self.down.bias)

        nn.init.kaiming_uniform_(self.dwconv.weight, a=math.sqrt(5))

        nn.init.zeros_(self.dwconv.bias)

        nn.init.zeros_(self.up.weight)

        nn.init.zeros_(self.up.bias)



    def forward(self, x):

        z = self.drop(self.act(self.down(x)))

        if z.ndim == 4:

            z_2d = z.permute(0, 3, 1, 2).contiguous()

            z_spatial = self.dwconv(z_2d).permute(0, 2, 3, 1).contiguous()

        else:

            B, L, r = z.shape

            H = W = int(math.isqrt(L))

            z_2d = z.view(B, H, W, r).permute(0, 3, 1, 2).contiguous()

            z_spatial = self.dwconv(z_2d).permute(0, 2, 3, 1).contiguous().view(B, L, r)

        return self.up(self.drop(self.act(z + z_spatial)))

class ForegroundAttentionPool(nn.Module):

    def __init__(self, dim, base_temp=0.20):

        super().__init__()

        self.dim = dim

        self.base_temp = base_temp

        self.query = nn.Parameter(torch.zeros(dim, 1))

        self.mix_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, x_stage):

        if x_stage.ndim == 4:

            x_stage = x_stage.flatten(1, 2)

        token_norm = x_stage.norm(p=2, dim=-1, keepdim=True)

        norm_z = (token_norm - token_norm.mean(dim=1, keepdim=True)) / (token_norm.std(dim=1, keepdim=True) + 1e-6)

        attn_logits = self.base_temp * norm_z + (x_stage @ self.query) / math.sqrt(self.dim)

        attn_w = F.softmax(attn_logits, dim=1)

        f_fg = (x_stage * attn_w).sum(dim=1)

        f_gap = x_stage.mean(dim=1)

        alpha = torch.sigmoid(self.mix_logit)

        return alpha * f_fg + (1.0 - alpha) * f_gap

class SwinPEFTModule(nn.Module):

    def __init__(self, stage_dims, stage_depths, stage_heads, peft_mode="P2_adapter",

                 fusion_mode="F3_multistage", window_size=7, rank=64, scale=0.8, dropout=0.1, swin_model=None):

        super().__init__()

        self.peft_mode = peft_mode

        self.fusion_mode = fusion_mode

        self.scale = scale

        self.act = nn.GELU()

        self.drop = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        if peft_mode == "P1_swin_apt":

            self.pk_list, self.pv_list = nn.ParameterList(), nn.ParameterList()

            for s_idx, d in enumerate(stage_depths):

                dim, nh = stage_dims[s_idx], stage_heads[s_idx]

                hd = dim // nh

                for _ in range(d):

                    p_k = nn.Parameter(torch.zeros(1, nh, window_size * window_size, hd))

                    p_v = nn.Parameter(torch.zeros(1, nh, window_size * window_size, hd))

                    nn.init.trunc_normal_(p_k, std=0.02)

                    nn.init.trunc_normal_(p_v, std=0.02)

                    self.pk_list.append(p_k)

                    self.pv_list.append(p_v)

        elif peft_mode == "P2_adapter":

            self.down_list, self.up_list = nn.ModuleList(), nn.ModuleList()

            for s_idx, d in enumerate(stage_depths):

                dim = stage_dims[s_idx]

                for _ in range(d):

                    down = nn.Linear(dim, rank, bias=False)

                    up = nn.Linear(rank, dim, bias=False)

                    nn.init.kaiming_uniform_(down.weight, a=math.sqrt(5))

                    nn.init.zeros_(up.weight)

                    self.down_list.append(down)

                    self.up_list.append(up)

        elif peft_mode == "P3_ssf":

            self.gamma_1, self.beta_1 = nn.ParameterList(), nn.ParameterList()

            self.gamma_2, self.beta_2 = nn.ParameterList(), nn.ParameterList()

            for s_idx, d in enumerate(stage_depths):

                dim = stage_dims[s_idx]

                for _ in range(d):

                    self.gamma_1.append(nn.Parameter(torch.ones(dim)))

                    self.beta_1.append(nn.Parameter(torch.zeros(dim)))

                    self.gamma_2.append(nn.Parameter(torch.ones(dim)))

                    self.beta_2.append(nn.Parameter(torch.zeros(dim)))

        elif peft_mode == "P4_convpass":

            self.convpass_list = nn.ModuleList()

            for s_idx, d in enumerate(stage_depths):

                dim = stage_dims[s_idx]

                for _ in range(d):

                    self.convpass_list.append(ConvPass2DAdapter(dim=dim, rank=rank, dropout=dropout))

        elif peft_mode in ["P5_hybrid_conv_rpb", "P6_fullstage_da"]:

            self.convpass_list = nn.ModuleList()

            self.delta_rpb = nn.ParameterList()

            for s_idx, d in enumerate(stage_depths):

                dim = stage_dims[s_idx]

                layer_idx = 2 + s_idx

                for b_idx in range(d):

                    self.convpass_list.append(ConvPass2DAdapter(dim=dim, rank=rank, dropout=dropout))

                    if swin_model is not None:

                        orig_bias = swin_model.layers[layer_idx].blocks[b_idx].attn.relative_position_bias_table

                        delta = nn.Parameter(torch.zeros_like(orig_bias))

                    else:

                        nh = stage_heads[s_idx]

                        num_rel_pos = (2 * window_size - 1) * (2 * window_size - 1)

                        delta = nn.Parameter(torch.zeros(num_rel_pos, nh))

                    self.delta_rpb.append(delta)

            if peft_mode == "P6_fullstage_da" and swin_model is not None:

                s1_dim = swin_model.layers[0].blocks[0].dim

                s2_dim = swin_model.layers[1].blocks[0].dim

                self.stem_gamma = nn.Parameter(torch.ones(s1_dim))

                self.stem_beta  = nn.Parameter(torch.zeros(s1_dim))

                self.early_g1, self.early_b1 = nn.ParameterList(), nn.ParameterList()

                self.early_g2, self.early_b2 = nn.ParameterList(), nn.ParameterList()

                for e_idx, e_dim in [(0, s1_dim), (1, s2_dim)]:

                    for _ in range(len(swin_model.layers[e_idx].blocks)):

                        self.early_g1.append(nn.Parameter(torch.ones(e_dim)))

                        self.early_b1.append(nn.Parameter(torch.zeros(e_dim)))

                        self.early_g2.append(nn.Parameter(torch.ones(e_dim)))

                        self.early_b2.append(nn.Parameter(torch.zeros(e_dim)))

        if fusion_mode == "F4_fg_multistage":

            self.fg_pool_s3 = ForegroundAttentionPool(stage_dims[0])

            self.fg_pool_s4 = ForegroundAttentionPool(stage_dims[1])

        self.init_state = {k: v.detach().clone() for k, v in self.state_dict().items()}

    def apply_ema_anchoring(self, alpha=0.7):

        with torch.no_grad():

            for k, v in self.named_parameters():

                if k in self.init_state:

                    v.data.copy_(alpha * v.data + (1.0 - alpha) * self.init_state[k].to(v.device))

    def grad_norm(self):

        sq_sum, has_grad = 0.0, False

        for p in self.parameters():

            if p.grad is not None:

                sq_sum += p.grad.detach().norm(2).item() ** 2

                has_grad = True

        return math.sqrt(sq_sum) if has_grad else 0.0

class SwinBackboneWrapper(nn.Module):

    def __init__(self, arch_key="swin_small", fusion_mode="F3_multistage"):

        super().__init__()

        timm_name = SWIN_ARCH_MAP[arch_key]

        self.swin = timm.create_model(timm_name, pretrained=True, num_classes=0)

        for p in self.swin.parameters():

            p.requires_grad = False

        self.swin.eval()

        self.register_buffer("imnet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))

        self.register_buffer("imnet_std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        self.fusion_mode = fusion_mode

        self.s3_dim = self.swin.layers[2].blocks[0].dim

        self.s4_dim = self.swin.layers[3].blocks[0].dim

        self.stage_dims   = [self.s3_dim, self.s4_dim]

        self.stage_depths = [len(self.swin.layers[2].blocks), len(self.swin.layers[3].blocks)]

        self.stage_heads  = [self.swin.layers[2].blocks[0].attn.num_heads,

                             self.swin.layers[3].blocks[0].attn.num_heads]

        if fusion_mode in ["F3_multistage", "F4_fg_multistage"]:

            self.out_features = self.s3_dim + self.s4_dim  # 1152d

        else:

            self.out_features = self.s4_dim                # 768d

    def build_peft(self, peft_mode="P2_adapter", rank=64, scale=0.8, dropout=0.1):

        if peft_mode == "P0_frozen":

            return None

        return SwinPEFTModule(

            self.stage_dims, self.stage_depths, self.stage_heads,

            peft_mode=peft_mode, fusion_mode=self.fusion_mode,

            rank=rank, scale=scale, dropout=dropout, swin_model=self.swin

        ).to(DEVICE)

    @staticmethod

    def _window_attn_with_apt(attn_mod, x_win, mask, pk, pv):

        B_, N, C = x_win.shape

        nh = attn_mod.num_heads

        hd = C // nh

        qkv = attn_mod.qkv(x_win).reshape(B_, N, 3, nh, hd).permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(0)

        k = k + pk.to(k.dtype)

        v = v + pv.to(v.dtype)

        q = q * attn_mod.scale

        attn = (q @ k.transpose(-2, -1)) + attn_mod._get_rel_pos_bias()

        if mask is not None:

            nW = mask.shape[0]

            attn = attn.view(-1, nW, nh, N, N) + mask.unsqueeze(1).unsqueeze(0)

            attn = attn.view(-1, nh, N, N)

        attn = attn_mod.attn_drop(attn.softmax(dim=-1))

        x_out = (attn @ v).transpose(1, 2).reshape(B_, N, C)

        return attn_mod.proj_drop(attn_mod.proj(x_out))

    def _run_early_block_with_ssf(self, blk, x, peft_mod, e_idx):

        g1, b1 = peft_mod.early_g1[e_idx], peft_mod.early_b1[e_idx]

        g2, b2 = peft_mod.early_g2[e_idx], peft_mod.early_b2[e_idx]

        orig_n1, orig_n2 = blk.norm1.forward, blk.norm2.forward

        blk.norm1.forward = lambda h, *a, _n1=orig_n1, _g1=g1, _b1=b1, **kw: _n1(h, *a, **kw) * _g1.to(h.dtype) + _b1.to(h.dtype)

        blk.norm2.forward = lambda h, *a, _n2=orig_n2, _g2=g2, _b2=b2, **kw: _n2(h, *a, **kw) * _g2.to(h.dtype) + _b2.to(h.dtype)

        try:

            return blk(x)

        finally:

            blk.norm1.forward = orig_n1

            blk.norm2.forward = orig_n2

    def _run_block_with_peft(self, blk, x, peft_mod, idx):

        if peft_mod is None:

            return blk(x)

        mode = peft_mod.peft_mode

        if mode == "P1_swin_apt":

            pk, pv = peft_mod.pk_list[idx], peft_mod.pv_list[idx]

            orig_fwd = blk.attn.forward

            blk.attn.forward = lambda x_w, *a, mask=None, _a=blk.attn, _pk=pk, _pv=pv, **kw: self._window_attn_with_apt(_a, x_w, mask, _pk, _pv)

            try:

                return blk(x)

            finally:

                blk.attn.forward = orig_fwd

        elif mode == "P2_adapter":

            orig_mlp_fwd = blk.mlp.forward

            down_layer = peft_mod.down_list[idx]

            up_layer   = peft_mod.up_list[idx]

            s          = peft_mod.scale

            act        = peft_mod.act

            drop       = peft_mod.drop

            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _d=down_layer, _u=up_layer, _s=s, _act=act, _drop=drop, **kw: _mlp(h, *a, **kw) + _s * _u(_drop(_act(_d(h))))

            try:

                return blk(x)

            finally:

                blk.mlp.forward = orig_mlp_fwd

        elif mode == "P3_ssf":

            g1, b1 = peft_mod.gamma_1[idx], peft_mod.beta_1[idx]

            g2, b2 = peft_mod.gamma_2[idx], peft_mod.beta_2[idx]

            orig_n1, orig_n2 = blk.norm1.forward, blk.norm2.forward

            blk.norm1.forward = lambda h, *a, _n1=orig_n1, _g1=g1, _b1=b1, **kw: _n1(h, *a, **kw) * _g1.to(h.dtype) + _b1.to(h.dtype)

            blk.norm2.forward = lambda h, *a, _n2=orig_n2, _g2=g2, _b2=b2, **kw: _n2(h, *a, **kw) * _g2.to(h.dtype) + _b2.to(h.dtype)

            try:

                return blk(x)

            finally:

                blk.norm1.forward = orig_n1

                blk.norm2.forward = orig_n2

        elif mode == "P4_convpass":

            orig_mlp_fwd = blk.mlp.forward

            cp = peft_mod.convpass_list[idx]

            s = peft_mod.scale

            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _cp=cp, _s=s, **kw: _mlp(h, *a, **kw) + _s * _cp(h)

            try:

                return blk(x)

            finally:

                blk.mlp.forward = orig_mlp_fwd

        elif mode in ["P5_hybrid_conv_rpb", "P6_fullstage_da"]:

            orig_mlp_fwd = blk.mlp.forward

            orig_get_rpb = blk.attn._get_rel_pos_bias

            cp = peft_mod.convpass_list[idx]

            d_rpb = peft_mod.delta_rpb[idx]

            s = peft_mod.scale

            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _cp=cp, _s=s, **kw: _mlp(h, *a, **kw) + _s * _cp(h)

            def _custom_rpb(_orig=orig_get_rpb, _attn=blk.attn, _drpb=d_rpb):

                bias = _drpb[_attn.relative_position_index.view(-1)].view(

                    _attn.window_size[0] * _attn.window_size[1],

                    _attn.window_size[0] * _attn.window_size[1], -1

                ).permute(2, 0, 1).contiguous().unsqueeze(0)

                return _orig() + bias.to(_attn.relative_position_bias_table.dtype)

            blk.attn._get_rel_pos_bias = _custom_rpb

            try:

                return blk(x)

            finally:

                blk.mlp.forward = orig_mlp_fwd

                blk.attn._get_rel_pos_bias = orig_get_rpb

    @staticmethod

    def _pool_tokens(x_stage, mode="gap"):

        if x_stage.ndim == 4:

            x_stage = x_stage.flatten(1, 2)

        if mode == "avgmax":

            return x_stage.mean(dim=1) + 0.35 * x_stage.amax(dim=1)

        return x_stage.mean(dim=1)

    def forward(self, x, peft_mod=None):

        x = (x - self.imnet_mean.to(x.dtype)) / self.imnet_std.to(x.dtype)

        use_early_da = (peft_mod is not None and getattr(peft_mod, "peft_mode", "") == "P6_fullstage_da")

        if not use_early_da:

            with torch.no_grad():

                x = self.swin.patch_embed(x)

                x = self.swin.layers[0](x)

                x = self.swin.layers[1](x)

        else:

            x = self.swin.patch_embed(x)

            x = x * peft_mod.stem_gamma.to(x.dtype) + peft_mod.stem_beta.to(x.dtype)

            e_idx = 0

            for s_early in [0, 1]:

                layer_e = self.swin.layers[s_early]

                if layer_e.downsample is not None:

                    x = layer_e.downsample(x)

                for blk in layer_e.blocks:

                    x = self._run_early_block_with_ssf(blk, x, peft_mod, e_idx)

                    e_idx += 1

        p_idx = 0

        layer3 = self.swin.layers[2]

        if layer3.downsample is not None:

            x = layer3.downsample(x)

        for blk in layer3.blocks:

            x = self._run_block_with_peft(blk, x, peft_mod, p_idx)

            p_idx += 1

        x_stage3 = x

        layer4 = self.swin.layers[3]

        x_stage3_normed = F.layer_norm(x_stage3, (self.s3_dim,))

        if layer4.downsample is not None:

            x = layer4.downsample(x)

        for blk in layer4.blocks:

            x = self._run_block_with_peft(blk, x, peft_mod, p_idx)

            p_idx += 1

        x_stage4 = self.swin.norm(x)

        if self.fusion_mode == "F1_gap":

            return self._pool_tokens(x_stage4, mode="gap")

        elif self.fusion_mode == "F2_avgmax":

            return self._pool_tokens(x_stage4, mode="avgmax")

        elif self.fusion_mode == "F3_multistage":

            f4 = self._pool_tokens(x_stage4, mode="gap")

            f3 = self._pool_tokens(x_stage3_normed, mode="gap")

            f3_rms = f3.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s3_dim) + 1e-6

            f4_rms = f4.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s4_dim) + 1e-6

            f3_aligned = f3 * (f4_rms / f3_rms)

            return torch.cat([f3_aligned, f4], dim=-1)

        elif self.fusion_mode == "F4_fg_multistage":

            if peft_mod is not None and hasattr(peft_mod, "fg_pool_s4"):

                f4 = peft_mod.fg_pool_s4(x_stage4)

                f3 = peft_mod.fg_pool_s3(x_stage3_normed)

            else:

                f4 = self._pool_tokens(x_stage4, mode="gap")

                f3 = self._pool_tokens(x_stage3_normed, mode="gap")

            f3_rms = f3.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s3_dim) + 1e-6

            f4_rms = f4.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s4_dim) + 1e-6

            f3_aligned = f3 * (f4_rms / f3_rms)

            return torch.cat([f3_aligned, f4], dim=-1)

# --- 3. FIRST-SESSION PEFT TRAINING FUNCTION ---

class CosineHead(nn.Module):

    def __init__(self, dim=768, num_classes=20, temp_scale=None, **kwargs):

        super().__init__()

        self.dim = dim

        self.norm = nn.LayerNorm(dim, elementwise_affine=False)

        self.fc = nn.Linear(dim, num_classes, bias=False)

        self.scale = math.sqrt(768.0 / float(dim))

    def forward(self, f):

        return (self.norm(f) * self.scale) @ F.normalize(self.fc.weight, dim=1).t()

def supcon_feature_loss(features, labels, temperature=0.10):

    f_norm = F.normalize(features.float(), p=2, dim=-1)

    B = f_norm.shape[0]

    if B <= 2:

        return torch.tensor(0.0, device=features.device)

    sim = (f_norm @ f_norm.T) / temperature

    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

    eye = torch.eye(B, dtype=torch.bool, device=features.device)

    pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)) & (~eye)

    exp_sim = torch.exp(sim) * (~eye).float()

    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    pos_cnt = pos_mask.float().sum(dim=1)

    valid = pos_cnt > 0

    if not valid.any():

        return torch.tensor(0.0, device=features.device)

    return (-(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_cnt[valid]).mean()

def train_first_session_peft(backbone_mod, peft_mod, task1_dataset, n_cls_t1, spec, in_features, use_supcon_mix=False):

    if peft_mod is None:

        return 0.0, 0.0

    

    is_vit = isinstance(backbone_mod, ViT)

    temp_head = CosineHead(dim=in_features, num_classes=n_cls_t1).to(DEVICE)



    if is_vit:

        peft_lr = spec.get("vit_prompt_lr", spec.get("prompt_lr", 0.005))

        head_lr = peft_lr

        bs1     = min(spec.get("vit_bs", 128), max(len(task1_dataset), 1))

        epochs  = spec.get("vit_epochs", spec.get("first_session_epochs", 50))

        warmup  = 0

        wd      = 0.0

        use_cos = False

        ema_val = spec.get("vit_ema", spec.get("ema_coeff", 0.70))

        opt = torch.optim.Adam([

            {"params": list(peft_mod.parameters()), "lr": peft_lr},

            {"params": list(temp_head.parameters()), "lr": head_lr}

        ], betas=(0.9, 0.999))

    else:

        peft_lr = spec.get("peft_lr", spec.get("swin_peft_lr", 0.003))

        head_lr = peft_lr * spec.get("head_lr_ratio", spec.get("swin_head_ratio", 2.5))

        bs1     = min(spec.get("batch_size", spec.get("swin_bs", 32)), max(len(task1_dataset), 1))

        epochs  = spec.get("epochs", spec.get("swin_epochs", 40))

        warmup  = spec.get("warmup_epochs", spec.get("swin_warmup", 2))

        wd      = spec.get("weight_decay", spec.get("swin_wd", 0.01))

        use_cos = spec.get("use_cosine_sched", spec.get("swin_cos", True))

        ema_val = spec.get("ema_coeff", spec.get("swin_ema", 0.90))

        opt = torch.optim.AdamW([

            {"params": list(peft_mod.parameters()), "lr": peft_lr, "weight_decay": wd},

            {"params": list(temp_head.parameters()), "lr": head_lr, "weight_decay": 0.0}

        ], betas=(0.9, 0.999))



    def lr_schedule_lambda(current_ep):

        ep = current_ep + 1

        if warmup > 0 and ep <= warmup:

            return max(float(ep) / float(warmup), 0.05)

        if not use_cos:

            return 1.0

        progress = float(ep - warmup) / float(max(epochs - warmup, 1))

        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))



    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_schedule_lambda)

    criterion = nn.CrossEntropyLoss()



    loader1 = DataLoader(task1_dataset, batch_size=bs1, shuffle=True, drop_last=False,

                         num_workers=NUM_WORKERS, pin_memory=True)

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE == "cuda")



    p_grads, c_grads = [], []

    for ep in range(1, epochs + 1):

        backbone_mod.eval()

        peft_mod.train()

        temp_head.train()

        tot_loss, ep_pg, ep_cg, n_b = 0.0, 0.0, 0.0, 0

        for x, y in loader1:

            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):

                feat = backbone_mod(x, prompt_fn=peft_mod.get) if is_vit else backbone_mod(x, peft_mod=peft_mod)

                logits = temp_head(feat)

                loss = criterion(logits, y)

                if use_supcon_mix and len(y) > 4:

                    l_supcon = supcon_feature_loss(feat, y, temperature=0.10)

                    perm = torch.randperm(len(y), device=y.device)

                    lam = float(np.random.beta(0.4, 0.4))

                    feat_mix = lam * feat + (1.0 - lam) * feat[perm]

                    logits_mix = temp_head(feat_mix)

                    l_mix = lam * criterion(logits_mix, y) + (1.0 - lam) * criterion(logits_mix, y[perm])

                    loss = 0.75 * loss + 0.25 * l_mix + 0.15 * l_supcon



            scaler.scale(loss).backward()

            scaler.unscale_(opt)

            pg = peft_mod.grad_norm()

            cg = temp_head.fc.weight.grad.detach().norm(2).item() if temp_head.fc.weight.grad is not None else 0.0

            scaler.step(opt)

            scaler.update()



            ep_pg += pg

            ep_cg += cg

            n_b += 1

            tot_loss += loss.item()



        scheduler.step()

        avg_pg = ep_pg / max(n_b, 1)

        avg_cg = ep_cg / max(n_b, 1)

        p_grads.append(avg_pg)

        c_grads.append(avg_cg)



        if ep == 1 or ep % 5 == 0 or ep == epochs:

            cur_lr = opt.param_groups[0]["lr"]

            tag = "ViT Baseline" if is_vit else "Swin J2"

            print(f"    [Task 1 PEFT - {tag}] Ep {ep:2d}/{epochs} | LR: {cur_lr:.5f} | Loss: {tot_loss/max(n_b, 1):.4f} | p_grad: {avg_pg:.4f} | c_grad: {avg_cg:.4f}")



    if not is_vit and hasattr(peft_mod, "apply_ema_anchoring"):

        peft_mod.apply_ema_anchoring(ema_val)



    for p in peft_mod.parameters():

        p.requires_grad = False

    peft_mod.eval()

    del temp_head

    if DEVICE == "cuda":

        torch.cuda.empty_cache()

    return round(float(np.mean(p_grads)), 4), round(float(np.mean(c_grads)), 4)


In [ ]:
# ==============================================================================
# CELL 6: CÁC CLASSIFICATION HEADS NGHIỆM ĐÓNG (v5 CHAMPION - MULTI-HEAD S3 INJECTION)
# ==============================================================================
class BaseRidgeHead(nn.Module):
    def __init__(self, M_unique, num_heads, head_dim, num_classes=200):
        super().__init__()
        self.M = M_unique
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.num_classes = num_classes
        self.optimal_lambda = 1000.0
        self.total_flops_accum = 0.0

    def find_optimal_lambda_fast(self, task_h_2d, task_y, candidate_lambdas, min_lambda=100.0):
        g = torch.Generator().manual_seed(1993)
        y_cpu = task_y.detach().cpu()
        tr_idx, val_idx = [], []
        for c in torch.unique(y_cpu).tolist():
            idx_c = (y_cpu == c).nonzero(as_tuple=True)[0]
            if len(idx_c) == 0:
                continue
            idx_c = idx_c[torch.randperm(len(idx_c), generator=g)]
            n_tr_c = max(int(len(idx_c) * 0.8), 1)
            tr_idx.append(idx_c[:n_tr_c])
            if len(idx_c) > n_tr_c:
                val_idx.append(idx_c[n_tr_c:])
        if len(tr_idx) == 0:
            tr_idx = torch.arange(len(task_y), device=task_h_2d.device)
            val_idx = tr_idx
        else:
            tr_idx = torch.cat(tr_idx).to(task_h_2d.device)
            val_idx = torch.cat(val_idx).to(task_h_2d.device) if len(val_idx) > 0 else tr_idx
        h_tr = task_h_2d[tr_idx].to(torch.float64)
        y_tr = F.one_hot(task_y[tr_idx], num_classes=self.num_classes).to(torch.float64)
        h_val = task_h_2d[val_idx].to(torch.float64)
        y_val = F.one_hot(task_y[val_idx], num_classes=self.num_classes).to(torch.float64)
        N_tr, D_h = h_tr.shape
        best_loss = float("inf")
        best_ridge = min_lambda
        if N_tr <= D_h:
            K_tr = h_tr @ h_tr.T
            K_val_tr = h_val @ h_tr.T
            I_N = torch.eye(N_tr, device=task_h_2d.device, dtype=torch.float64)
            for r in candidate_lambdas:
                if r < min_lambda:
                    continue
                try:
                    L = torch.linalg.cholesky(K_tr + r * I_N)
                    alpha_r = torch.cholesky_solve(y_tr, L)
                    pred = K_val_tr @ alpha_r
                    loss = F.mse_loss(pred, y_val).item()
                    if loss < best_loss:
                        best_loss = loss
                        best_ridge = r
                except Exception:
                    continue
        else:
            G_tr = h_tr.T @ h_tr
            Q_tr = h_tr.T @ y_tr
            I_D = torch.eye(D_h, device=task_h_2d.device, dtype=torch.float64)
            for r in candidate_lambdas:
                if r < min_lambda:
                    continue
                try:
                    L = torch.linalg.cholesky(G_tr + r * I_D)
                    W_r = torch.cholesky_solve(Q_tr, L)
                    pred = h_val @ W_r
                    loss = F.mse_loss(pred, y_val).item()
                    if loss < best_loss:
                        best_loss = loss
                        best_ridge = r
                except Exception:
                    continue
        return best_ridge


class LinearRidgeHead(BaseRidgeHead):
    def __init__(self, in_features=768, num_classes=200):
        super().__init__(in_features, 1, in_features, num_classes)
        self.in_features = in_features
        self.register_buffer("G", torch.zeros(1, in_features, in_features, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(1, in_features, num_classes, dtype=torch.float32))
        self.register_buffer("weight", torch.zeros(num_classes, in_features, dtype=torch.float32))

    def expand_features(self, f):
        return f.float()

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        h = self.expand_features(task_f)
        N = h.shape[0]
        Y = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.optimal_lambda = self.find_optimal_lambda_fast(h, task_y, candidate_lambdas, min_lambda=10.0)
        self.G[0] += h.T @ h
        self.Q[0] += h.T @ Y
        I_D = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        G_reg = self.G[0].to(torch.float64) + self.optimal_lambda * I_D
        Q_64 = self.Q[0].to(torch.float64)
        L, info = torch.linalg.cholesky_ex(G_reg)
        W_opt = torch.cholesky_solve(Q_64, L) if info.item() == 0 else torch.linalg.solve(G_reg, Q_64)
        self.weight.copy_(W_opt.T.float())
        self.total_flops_accum += (2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes +
                                   (1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes)

    def forward(self, f):
        return self.expand_features(f) @ self.weight.T


class RanPACHead(BaseRidgeHead):
    def __init__(self, in_features=768, num_classes=200, M=10000, seed=1993):
        super().__init__(M, 1, M, num_classes)
        self.in_features = in_features
        self.register_buffer("G", torch.zeros(1, M, M, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(1, M, num_classes, dtype=torch.float32))
        self.register_buffer("weight", torch.zeros(num_classes, M, dtype=torch.float32))
        g = torch.Generator().manual_seed(seed)
        self.register_buffer("W_rand", torch.randn(in_features, M, generator=g))

    def expand_features(self, f):
        return F.relu(f.float() @ self.W_rand)

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        h = self.expand_features(task_f)
        N = h.shape[0]
        Y = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.optimal_lambda = self.find_optimal_lambda_fast(h, task_y, candidate_lambdas, min_lambda=1.0)
        self.G[0] += h.T @ h
        self.Q[0] += h.T @ Y
        I_D = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        G_reg = self.G[0].to(torch.float64) + self.optimal_lambda * I_D
        Q_64 = self.Q[0].to(torch.float64)
        L, info = torch.linalg.cholesky_ex(G_reg)
        W_opt = torch.cholesky_solve(Q_64, L) if info.item() == 0 else torch.linalg.solve(G_reg, Q_64)
        self.weight.copy_(W_opt.T.float())
        self.total_flops_accum += (2.0 * N * self.in_features * self.head_dim +
                                   2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes +
                                   (1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes)

    def forward(self, f):
        return self.expand_features(f) @ self.weight.T


class ConfigurableSOKACHead(BaseRidgeHead):
    # SO-KAC-3ReLU v5 Champion (Calibrated Sobolev Kolmogorov-Arnold Classifier):
    # 1. Multi-Head Orthogonal Stage-3 Injection (P_s3_heads: 384 -> 768) bảo toàn 100% ngữ nghĩa Stage 4.
    # 2. Monotonic Multi-Order Sobolev B-Spline Basis (s0 + 0.45*s1 + 0.25*s2 + 0.20*s_neg).
    # 3. Hybrid Calibration (70% raw + 30% L2 hypersphere) + Implicit Bias (+0.10 * scale).
    # 4. Class-Prior Balancing (cb_gamma) & Task Weight Aligning (wa_beta) bảo vệ dải biên độ liên task.
    # 5. Khóa cứng RAM ma trận Gram G = 6 x 2304 x 2304 x 4B = 121.50 MB.
    def __init__(
        self,
        in_features=768,
        num_classes=200,
        num_heads=6,
        orth_dim=1536,
        cb_gamma=0.18,
        wa_beta=0.30,
        seed=1993
    ):
        self.skip_dim = 768
        head_dim = self.skip_dim + orth_dim
        super().__init__(self.skip_dim + num_heads * orth_dim, num_heads, head_dim, num_classes)
        self.in_features = in_features
        self.orth_dim = orth_dim
        self.class_balance_gamma = cb_gamma
        self.task_wa_beta = wa_beta
        self.alpha_spline = 0.45
        self.kac_gain = 1.45
        self.task_count = 0
        self.task_class_ranges = []

        self.register_buffer("global_rms", torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer("G", torch.zeros(self.num_heads, self.head_dim, self.head_dim, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(self.num_heads, self.head_dim, num_classes, dtype=torch.float32))
        self.register_buffer("class_counts", torch.zeros(num_classes, dtype=torch.float64))
        self.register_buffer("weight", torch.zeros(self.num_heads, num_classes, self.head_dim, dtype=torch.float32))

        g = torch.Generator().manual_seed(seed)
        total_orth_cols = self.num_heads * self.orth_dim
        num_qr_blocks = (total_orth_cols + in_features - 1) // in_features
        orth_blocks = []
        for _ in range(num_qr_blocks):
            Z = torch.randn(in_features, in_features, generator=g)
            Q_qr, R_qr = torch.linalg.qr(Z)
            d_sign = torch.diag(R_qr).sign()
            d_sign[d_sign == 0] = 1.0
            orth_blocks.append(Q_qr * d_sign.unsqueeze(0))
        W_all = torch.cat(orth_blocks, dim=1)[:, :total_orth_cols]
        self.register_buffer("Q_orth", W_all)

        if in_features > self.skip_dim:
            s3_dim = in_features - self.skip_dim  # 384
            s3_blocks = []
            for _ in range(self.num_heads):
                Z_s3 = torch.randn(self.skip_dim, self.skip_dim, generator=g)
                Q_s3, R_s3 = torch.linalg.qr(Z_s3)
                d_s3 = torch.diag(R_s3).sign()
                d_s3[d_s3 == 0] = 1.0
                P_h = (Q_s3 * d_s3.unsqueeze(0))[:s3_dim, :] * math.sqrt(self.skip_dim / float(s3_dim))
                s3_blocks.append(P_h)
            self.register_buffer("P_s3_heads", torch.stack(s3_blocks, dim=0))  # (H, 384, 768)
        else:
            self.P_s3_heads = None

        k1 = torch.linspace(0.15, 0.70, self.orth_dim)
        k2 = torch.linspace(0.70, 1.45, self.orth_dim)
        self.register_buffer("knot_1", k1.view(1, 1, self.orth_dim))
        self.register_buffer("knot_2", k2.view(1, 1, self.orth_dim))

    @torch.no_grad()
    def init_global_scale(self, task_f):
        if self.global_rms.item() == 0.0:
            rms_val = (task_f.float().norm(p=2, dim=-1) / math.sqrt(self.in_features)).mean().item()
            self.global_rms.fill_(max(rms_val, 1e-4))

    def expand_features(self, f):
        f_32 = f.float()
        B = f_32.shape[0]
        scale = self.global_rms.clamp(min=1e-4)
        f_unit = F.normalize(f_32, p=2, dim=-1) * (math.sqrt(float(self.in_features)) * scale)
        f_cal = 0.70 * f_32 + 0.30 * f_unit
        f_scaled = f_cal / scale
        z = (f_scaled @ self.Q_orth).view(B, self.num_heads, self.orth_dim)

        s0 = F.relu(z)
        s1 = F.relu(z - self.knot_1)
        s2 = F.relu(z - self.knot_2)
        if self.in_features == self.skip_dim:
            psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2))
        else:
            s_neg = F.relu(-z - self.knot_1)
            psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2 + 0.20 * s_neg))
        psi_orth = (psi_raw - psi_raw.mean(dim=-1, keepdim=True)) + (0.10 * scale)

        if self.in_features == self.skip_dim:
            f_expand = f_cal.unsqueeze(1).expand(B, self.num_heads, self.skip_dim)
        else:
            s3_dim = self.in_features - self.skip_dim
            f_s3 = f_cal[:, :s3_dim]       # (B, 384)
            f_s4 = f_cal[:, s3_dim:]       # (B, 768)
            f_s3_proj = torch.einsum("bd,hdk->bhk", f_s3, self.P_s3_heads)  # (B, H, 768)
            gamma_s3 = 0.45
            f_expand = (f_s4.unsqueeze(1) + gamma_s3 * f_s3_proj) / math.sqrt(1.0 + gamma_s3 ** 2)

        return torch.cat([f_expand, psi_orth], dim=-1)

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        if candidate_lambdas is None:
            candidate_lambdas = [10.0**p for p in range(-8, 9)]
        self.init_global_scale(task_f)
        if task_range is not None:
            t_lo, t_hi = task_range
        else:
            t_lo, t_hi = int(task_y.min().item()), int(task_y.max().item()) + 1
        if (t_lo, t_hi) not in self.task_class_ranges:
            self.task_class_ranges.append((t_lo, t_hi))

        task_h = self.expand_features(task_f)
        N = task_h.shape[0]
        task_h_32 = task_h.permute(1, 0, 2)
        task_Y_32 = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.task_count += 1

        is_imbal = False
        if len(self.task_class_ranges) > 0:
            seen = self.class_counts > 0
            if seen.any() and (self.class_counts[seen].std() / (self.class_counts[seen].mean() + 1e-8)).item() > 0.08:
                is_imbal = True

        rms_ratio_sq = min(self.global_rms.item() ** 2, 1.0)
        min_lam = (200.0 if is_imbal else 30.0) * rms_ratio_sq
        raw_lambda = self.find_optimal_lambda_fast(task_h[:, 0, :], task_y, candidate_lambdas, min_lambda=min_lam)
        if self.task_count == 1:
            self.base_lambda = raw_lambda
        else:
            self.base_lambda = (self.base_lambda ** 0.85) * (raw_lambda ** 0.15)
        self.optimal_lambda = self.base_lambda * (self.task_count ** 0.12)

        self.G += torch.bmm(task_h_32.transpose(1, 2), task_h_32)
        self.Q += torch.matmul(task_h_32.transpose(1, 2), task_Y_32.unsqueeze(0))
        self.class_counts += task_Y_32.to(torch.float64).sum(dim=0)

        seen_mask = self.class_counts > 0
        cls_scale = torch.ones_like(self.class_counts)
        is_imbalanced_dataset = False
        if seen_mask.any():
            counts_seen = self.class_counts[seen_mask]
            cv_counts = (counts_seen.std() / (counts_seen.mean() + 1e-8)).item()
            if cv_counts > 0.08:
                is_imbalanced_dataset = True
                if self.class_balance_gamma > 0.0:
                    mean_count = counts_seen.mean()
                    cls_scale[seen_mask] = (mean_count / counts_seen).pow(self.class_balance_gamma)

        I_mat = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        W_list = []
        for h in range(self.num_heads):
            G_reg = self.G[h].to(torch.float64) + self.optimal_lambda * I_mat
            Q_h = self.Q[h].to(torch.float64) * cls_scale.unsqueeze(0)
            L_h, info = torch.linalg.cholesky_ex(G_reg)
            if info.item() == 0:
                W_h = torch.cholesky_solve(Q_h, L_h)
            else:
                W_h = torch.linalg.solve(G_reg, Q_h)
            W_list.append(W_h.float())
        W_opt = torch.stack(W_list, dim=0)
        W_t = W_opt.transpose(1, 2)

        eff_beta = self.task_wa_beta if is_imbalanced_dataset else 0.0
        if len(self.task_class_ranges) > 1 and eff_beta > 0.0:
            task_norms = []
            for (lo, hi) in self.task_class_ranges:
                w_task = W_t[:, lo:hi, :]
                rms_task = w_task.pow(2).sum(dim=-1, keepdim=True).mean(dim=1, keepdim=True).sqrt()
                task_norms.append(rms_task)
            mean_task_norm = torch.stack(task_norms, dim=0).mean(dim=0)
            for (lo, hi), r_t in zip(self.task_class_ranges, task_norms):
                scale_t = (mean_task_norm / (r_t + 1e-8)).pow(eff_beta)
                W_t[:, lo:hi, :] *= scale_t

        self.weight.copy_(W_t)
        self.total_flops_accum += (2.0 * N * self.in_features * (self.num_heads * self.orth_dim) +
                                   self.num_heads * (2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes) +
                                   self.num_heads * ((1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes))

    def forward(self, f):
        h = self.expand_features(f)
        return torch.einsum("bhd,hcd->bc", h, self.weight) / self.num_heads


def build_classification_head(head_name, in_features, num_classes, seed=1993, cb_gamma=0.18, wa_beta=0.30):
    if head_name in ["B0_linear_ridge", "Linear_Ridge"]:
        return LinearRidgeHead(in_features=in_features, num_classes=num_classes)
    elif head_name in ["B2_ranpac_10k", "RanPAC_10k"]:
        return RanPACHead(in_features=in_features, num_classes=num_classes, M=10000, seed=seed)
    elif head_name in ["B3_ranpac_ram5640", "RanPAC_IsoRAM"]:
        return RanPACHead(in_features=in_features, num_classes=num_classes, M=5640, seed=seed)
    else:  # SO_KAC_3ReLU
        return ConfigurableSOKACHead(in_features=in_features, num_classes=num_classes, cb_gamma=cb_gamma, wa_beta=wa_beta, seed=seed)


In [ ]:
# ==============================================================================
# CELL 7: METRICS CHẨN ĐOÁN & ENGINE ĐÁNH GIÁ CONTINUAL LEARNING CHUẨN
# ==============================================================================
@torch.no_grad()
def compute_feature_eff_rank(f_mat):
    f_c = f_mat - f_mat.mean(dim=0, keepdim=True)
    cov = (f_c.T @ f_c) / max(len(f_mat) - 1, 1)
    eigvals = torch.linalg.eigvalsh(cov.to(torch.float64)).clamp(min=0.0)
    S = eigvals.sqrt()
    S = S / (S.sum() + 1e-12)
    entropy = -(S * torch.log(S + 1e-12)).sum()
    return float(torch.exp(entropy).item())
@torch.no_grad()
def compute_gram_eff_rank(G_head0):
    eigvals = torch.linalg.eigvalsh(G_head0.to(torch.float64)).clamp(min=0.0)
    S = eigvals.sqrt()
    S = S / (S.sum() + 1e-12)
    entropy = -(S * torch.log(S + 1e-12)).sum()
    return float(torch.exp(entropy).item())
@torch.no_grad()
def compute_fisher_discriminant_ratios(ev_f, ev_y, task1_hi):
    def _calc_fdr(f_subset, y_subset):
        classes = torch.unique(y_subset)
        if len(classes) <= 1:
            return 0.0
        f_subset = f_subset.to(torch.float64)
        mu_total = f_subset.mean(dim=0)
        s_b_trace = 0.0
        s_w_trace = 0.0
        for c in classes:
            f_c = f_subset[y_subset == c]
            n_c = len(f_c)
            if n_c == 0:
                continue
            mu_c = f_c.mean(dim=0)
            diff_b = mu_c - mu_total
            s_b_trace += n_c * torch.dot(diff_b, diff_b).item()
            diff_w = f_c - mu_c
            s_w_trace += (diff_w ** 2).sum().item()
        return float(s_b_trace / (s_w_trace + 1e-12))
    mask_t1 = (ev_y < task1_hi)
    mask_unseen = (ev_y >= task1_hi)
    fdr_t1 = _calc_fdr(ev_f[mask_t1], ev_y[mask_t1]) if mask_t1.any() else 0.0
    fdr_unseen = _calc_fdr(ev_f[mask_unseen], ev_y[mask_unseen]) if mask_unseen.any() else 0.0
    return round(fdr_t1, 3), round(fdr_unseen, 3)
@torch.no_grad()
def evaluate_head(head, cached_ev_f, cached_ev_y, n_seen_tasks, bounds, eff_rank_backbone=0.0, compute_phi_rank=False):
    hi = bounds[n_seen_tasks - 1][1]
    mask_seen = (cached_ev_y < hi)
    f_all = cached_ev_f[mask_seen].to(DEVICE)
    y = cached_ev_y[mask_seen].to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t_inf0 = time.perf_counter()
    B_eval = 512
    logits_list = []
    for i in range(0, len(f_all), B_eval):
        logits_list.append(head(f_all[i:i + B_eval])[:, :hi])
    logits = torch.cat(logits_list, dim=0)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    infer_time_ms = (time.perf_counter() - t_inf0) * 1000.0
    if hasattr(head, "Q_orth"):
        in_dim = min(getattr(head, "in_features", 768), head.Q_orth.shape[1])
        scale = head.global_rms.clamp(min=1e-4)
        f_u = F.normalize(f_all.float(), p=2, dim=-1) * (math.sqrt(float(in_dim)) * scale)
        f_cal_check = 0.70 * f_all.float() + 0.30 * f_u
        z_check = (f_cal_check / scale) @ head.Q_orth[:, :in_dim]
        u_out = ((z_check < -2.0) | (z_check > 2.0)).float().mean().item() * 100.0
    else:
        u_out = 0.0
    correct_ta = 0
    accs = []
    preds = logits.argmax(dim=1)
    for j in range(n_seen_tasks):
        t_lo, t_hi = bounds[j]
        mask = (y >= t_lo) & (y < t_hi)
        if mask.sum() > 0:
            pred_ta = logits[mask, t_lo:t_hi].argmax(dim=1) + t_lo
            correct_ta += (pred_ta == y[mask]).sum().item()
            accs.append(100.0 * (preds[mask] == y[mask]).float().mean().item())
        else:
            accs.append(0.0)
    acc_ta = 100.0 * correct_ta / max(len(y), 1)
    overall = 100.0 * (preds == y).float().mean().item()
    eff_rank_phi = compute_gram_eff_rank(head.G[0]) if compute_phi_rank else 0.0
    diag = {
        "eff_rank": round(eff_rank_backbone, 2),
        "eff_rank_phi": round(eff_rank_phi, 2),
        "u_out_of_grid": round(u_out, 2),
        "acc_ta": round(acc_ta, 2),
        "optimal_lambda": float(head.optimal_lambda),
        "infer_time_ms": round(infer_time_ms, 2),
    }
    return accs, overall, diag
def get_hardware_stats(head):
    gram_ram_mb = (head.G.numel() * head.G.element_size()) / (1024**2)
    total_buf_mb = sum(b.numel() * b.element_size() for b in head.buffers()) / (1024**2)
    cholesky_macs_per_task = head.num_heads * ((1.0 / 6.0) * (head.head_dim ** 3))
    return dict(
        M_unique=int(head.M), num_heads=int(head.num_heads), head_dim=int(head.head_dim),
        config_str=f"{head.num_heads}x{head.head_dim}",
        gram_ram_mb=round(gram_ram_mb, 2), total_buf_mb=round(total_buf_mb, 2),
        cholesky_gmacs=round(cholesky_macs_per_task / 1e9, 3),
        total_gflops_10t=round(head.total_flops_accum / 1e9, 2),
    )
def summarize(R):
    Tn = R.shape[0]
    A = [float(np.mean(R[t, :t + 1])) for t in range(Tn)]
    fgt = float(np.mean([R[j, j] - R[Tn - 1, j] for j in range(Tn - 1)])) if Tn > 1 else 0.0
    return dict(A_t=A, last=round(A[-1], 2), avg=round(float(np.mean(A)), 2), forgetting=round(fgt, 2))
def extract_features_loader(backbone, peft_mod, data_loader, use_flip_gram=False, power_beta=0.85):
    '''
    Nếu use_flip_gram=True (Exp6a..6d):
      - Trung bình cộng đặc trưng của ảnh gốc và ảnh lật ngang (Horizontal Flip) để khử nhiễu góc chụp Few-Shot.
      - Áp dụng biến đổi Tukey Power-Norm nhẹ (sign(f) * |f|^0.85) giúp phân phối đặc trưng Swin chuẩn Gauss hơn.
    '''
    feats, targets = [], []
    backbone.eval()
    if peft_mod is not None:
        peft_mod.eval()
    is_vit = isinstance(backbone, ViT)
    prompt_fn = peft_mod.get if (is_vit and peft_mod is not None) else None
    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
                if is_vit:
                    f = backbone(x, prompt_fn=prompt_fn)
                else:
                    f = backbone(x, peft_mod=peft_mod)
                    if use_flip_gram:
                        x_flip = torch.flip(x, dims=[-1])
                        f_flip = backbone(x_flip, peft_mod=peft_mod)
                        f = 0.5 * (f + f_flip)
            f_32 = f.float()
            if use_flip_gram and power_beta != 1.0:
                f_32 = torch.sign(f_32) * (f_32.abs() + 1e-8).pow(power_beta)
            feats.append(f_32.cpu())
            targets.append(y)
    return torch.cat(feats, dim=0), torch.cat(targets, dim=0)


In [ ]:
# ==============================================================================
# CELL 8: THỰC THI PHASE 1 (VAL ABLATION) & PHASE 2 (FINAL TEST 100% TRAIN)
# ==============================================================================
def build_head(head_name, in_features, num_classes, seed=1993, cb_gamma=0.18, wa_beta=0.30):
    return build_classification_head(head_name, in_features, num_classes, seed=seed, cb_gamma=cb_gamma, wa_beta=wa_beta)

def train_and_eval_continual(head, cached_train_tasks, cached_ev_f, cached_ev_y, bounds, ridge_lambdas, verbose=False):
    Tn = len(bounds)
    R_mat = np.zeros((Tn, Tn), dtype=np.float32)
    task_logs, solve_times = [], []
    for t in range(Tn):
        t_f, t_y, lo, hi = cached_train_tasks[t]
        t_f, t_y = t_f.to(DEVICE), t_y.to(DEVICE)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        head.update_and_solve(t_f, t_y, ridge_lambdas, task_range=(lo, hi))
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        solve_times.append(time.perf_counter() - t0)
        accs, overall, diag = evaluate_head(
            head, cached_ev_f, cached_ev_y, t + 1, bounds
        )
        R_mat[t, :t + 1] = accs
        task_logs.append(dict(task=t + 1, A_t=float(np.mean(accs)), sample_acc=overall, diag=diag))
    res = summarize(R_mat)
    hw = get_hardware_stats(head)
    last_diag = task_logs[-1]["diag"]
    res.update(
        last_acc=res["last"],
        avg_acc=res["avg"],
        fgt=res["forgetting"],
        acc_ta=last_diag["acc_ta"],
        eff_rank=last_diag["eff_rank"],
        eff_rank_phi=last_diag["eff_rank_phi"],
        u_out_of_grid=last_diag["u_out_of_grid"],
        ram_mb=hw["gram_ram_mb"],
        mean_solve_s=round(float(np.mean(solve_times)), 3)
    )
    return res

def format_swin_table(summary_dict):
    headers = [
        "Experiment ID", "Backbone", "Fusion", "PEFT", "Head",
        "Last A_T (%)", "Avg A_avg (%)", "Fgt (%)", "TA (%)", "eff_rank", "FDR (T1/Unseen)", "RAM (MB)"
    ]
    rows = []
    for k, v in summary_dict.items():
        rows.append([
            k, v.get("backbone", "-"), v.get("fusion", "-"), v.get("peft", "-"), v.get("head", "-"),
            f"{v.get('last_mean', 0.0):.2f} ± {v.get('last_std', 0.0):.2f}",
            f"{v.get('avg_mean', 0.0):.2f} ± {v.get('avg_std', 0.0):.2f}",
            f"{v.get('fgt_mean', 0.0):.2f}",
            f"{v.get('ta_mean', 0.0):.2f}",
            f"{v.get('eff_rank_mean', 0.0):.1f}",
            f"{v.get('fdr_t1_mean', 0.0):.2f}/{v.get('fdr_unseen_mean', 0.0):.2f}",
            f"{v.get('ram_mb', 0.0):.1f}"
        ])
    col_w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    h_str = " | ".join(headers[i].ljust(col_w[i]) for i in range(len(headers)))
    sep = "-+-".join("-" * col_w[i] for i in range(len(headers)))
    lines = [h_str, sep]
    for r in rows:
        lines.append(" | ".join(str(r[i]).ljust(col_w[i]) for i in range(len(headers))))
    return "\n".join(lines)

for d_key in AVAILABLE_DATASETS:
    spec = DATASET_SPECS[d_key]
    d_name = spec["dataset_name"]
    d_out = os.path.join(MASTER_OUT_DIR, d_name)
    os.makedirs(d_out, exist_ok=True)
    print(f"\n{'='*100}\n🚀 BẮT ĐẦU EXPERIMENTS TRÊN DATASET: {d_name} (Lớp: {spec['n_classes']})\n{'='*100}")
    train_tf, eval_tf = build_transforms(spec["aug_type"])
    raw_train, raw_test = load_raw_dataset_samples(d_key, spec)
    is_cifar = (d_key == "cifar100")

    p_lr = spec.get("peft_lr", spec.get("swin_peft_lr", 0.003))
    h_lr = p_lr * spec.get("head_lr_ratio", spec.get("swin_head_ratio", 2.5))
    bs_val = spec.get("batch_size", spec.get("swin_bs", 32))
    ep_cnt = spec.get("epochs", spec.get("swin_epochs", 40))
    rk_val = spec.get("rank", spec.get("swin_rank", 64))
    sc_val = spec.get("scale", spec.get("swin_scale", 0.8))
    ev_bs  = spec.get("eval_batch_size", 128)
    print(f"📌 [TASK 1 PEFT]: LR={p_lr:.5f} | Head_LR={h_lr:.5f} | BS={bs_val} | Epochs={ep_cnt} | Rank={rk_val} | Scale={sc_val} | EMA={spec.get('ema_coeff', 0.90)} | cb_gamma={spec.get('cb_gamma', 0.0)} | wa_beta={spec.get('wa_beta', 0.0)}")
    print(f"📌 [FEATURE EXTRACTION PIPELINE]: Eval_BS={ev_bs} (Trích xuất đặc trưng tăng tốc từ RAM Zero-Disk I/O)")

    # ==========================================================================
    # GIAI ĐOẠN 1: VAL ABLATION (80/20 DISJOINT, 3 SEEDS) — ZERO TEST-LEAKAGE
    # ==========================================================================
    if RUN_VAL_ABLATION:
        print(f"\n{'='*100}\n🔬 GIAI ĐOẠN 1: VAL ABLATION (80/20 DISJOINT, 3 SEEDS) — ZERO TEST-LEAKAGE: {d_name}\n{'='*100}")
        active_exps = OrderedDict((k, v) for k, v in SWIN_ABLATION_EXPERIMENTS.items() if k in ACTIVE_ABLATION_EXPERIMENTS)
        val_runs = OrderedDict()
        for exp_id in active_exps.keys():
            for h_name in ACTIVE_HEADS:
                key = exp_id if len(ACTIVE_HEADS) == 1 else f"{exp_id}_{h_name}"
                val_runs[key] = []

        for seed in SEEDS_TO_RUN:
            print(f"\n--- [VAL - SEED {seed}] Tách Fold {seed % 5} làm Val Disjoint ---")
            seed_everything(seed)
            class_order, remap, bounds, tr_idx, va_idx = setup_cil_seed(
                raw_train, seed, spec["n_classes"], spec["task_sizes"], is_cifar=is_cifar
            )

            ev_preloaded = precompute_eval_tensors(raw_train, va_idx, remap, eval_tf, is_cifar=is_cifar, desc=f"Val Fold {seed % 5}")
            ev_loader = DataLoader(ev_preloaded, batch_size=ev_bs, shuffle=False, num_workers=0, pin_memory=True)

            targets_all = raw_train[1] if is_cifar else [y for _, y in raw_train]
            cached_train_loaders = {}
            for t, (lo, hi) in enumerate(bounds):
                t_train_idx = [i for i in tr_idx if lo <= remap[int(targets_all[i])] < hi]
                t_preloaded = precompute_eval_tensors(raw_train, t_train_idx, remap, eval_tf, is_cifar=is_cifar, desc=f"Train Task {t+1}")
                cached_train_loaders[t] = (DataLoader(t_preloaded, batch_size=ev_bs, shuffle=False, num_workers=0, pin_memory=True), lo, hi)

            feature_cache = {}
            trained_peft_cache = {}

            for exp_id, exp_spec in active_exps.items():
                bb_choice  = exp_spec["backbone"]
                fs_choice  = exp_spec["fusion"]
                eval_fs    = exp_spec.get("eval_fusion", fs_choice)
                pf_choice  = exp_spec["peft"]
                use_flip   = exp_spec.get("flip_gram", False)
                use_sc_mix = exp_spec.get("supcon_mix", False)

                peft_train_key = (bb_choice, fs_choice, pf_choice, use_sc_mix)
                cache_key      = (bb_choice, eval_fs, pf_choice, use_sc_mix, use_flip)

                if cache_key not in feature_cache:
                    if peft_train_key not in trained_peft_cache:
                        if bb_choice == "vit_base":
                            backbone = load_pretrained_vit().to(DEVICE)
                            peft_mod = APTPrompt(depth=12, dim=768, heads=12, ema_coeff=spec.get("vit_ema", 0.70)).to(DEVICE)
                            in_feat_dim = 768
                        else:
                            backbone = SwinBackboneWrapper(arch_key=bb_choice, fusion_mode=fs_choice).to(DEVICE)
                            swin_rank  = exp_spec.get("rank", spec.get("rank", 64))
                            swin_scale = exp_spec.get("scale", spec.get("scale", 0.8))
                            swin_drop  = exp_spec.get("adapter_dropout", spec.get("adapter_dropout", 0.1))
                            peft_mod = backbone.build_peft(
                                peft_mode=pf_choice, rank=swin_rank, scale=swin_scale, dropout=swin_drop
                            )
                            in_feat_dim = backbone.out_features

                        t1_lo, t1_hi = bounds[0]
                        t1_train_idx = [i for i in tr_idx if t1_lo <= remap[int(targets_all[i])] < t1_hi]
                        if peft_mod is not None:
                            t1_ds = CachedTrainSubset(raw_train, t1_train_idx, remap, train_tf, is_cifar=is_cifar)
                            p_grad, c_grad = train_first_session_peft(
                                backbone, peft_mod, t1_ds, t1_hi - t1_lo, spec, in_feat_dim, use_supcon_mix=use_sc_mix
                            )
                            del t1_ds
                        else:
                            p_grad, c_grad = 0.0, 0.0
                        trained_peft_cache[peft_train_key] = (backbone, peft_mod, in_feat_dim, p_grad)

                    backbone, peft_mod, in_feat_dim, p_grad = trained_peft_cache[peft_train_key]
                    t1_lo, t1_hi = bounds[0]

                    cached_ev_f, cached_ev_y = extract_features_loader(backbone, peft_mod, ev_loader, use_flip_gram=use_flip, override_fusion=eval_fs)
                    fdr_t1, fdr_unseen = compute_fisher_discriminant_ratios(cached_ev_f, cached_ev_y, t1_hi)

                    cached_train_tasks, eff_ranks_backbone = [], []
                    for t in range(len(bounds)):
                        t_ld, lo, hi = cached_train_loaders[t]
                        t_f, t_y = extract_features_loader(backbone, peft_mod, t_ld, use_flip_gram=use_flip, override_fusion=eval_fs)
                        cached_train_tasks.append((t_f, t_y, lo, hi))
                        mask_seen = (cached_ev_y < hi)
                        eff_ranks_backbone.append(compute_feature_eff_rank(cached_ev_f[mask_seen]))

                    feature_cache[cache_key] = (cached_train_tasks, cached_ev_f, cached_ev_y, eff_ranks_backbone, p_grad, fdr_t1, fdr_unseen, in_feat_dim)

                cached_train_tasks, cached_ev_f, cached_ev_y, eff_ranks_backbone, p_grad, fdr_t1, fdr_unseen, in_feat_dim = feature_cache[cache_key]

                for h_name in ACTIVE_HEADS:
                    cb_val = 0.32 if (bb_choice == "vit_base" and spec.get("cb_gamma", 0.0) > 0) else spec.get("cb_gamma", 0.0)
                    wa_val = 0.80 if (bb_choice == "vit_base" and spec.get("wa_beta", 0.0) > 0) else spec.get("wa_beta", 0.0)
                    head_inst = build_head(h_name, in_feat_dim, spec["n_classes"], seed=1993, cb_gamma=cb_val, wa_beta=wa_val).to(DEVICE)
                    res = train_and_eval_continual(
                        head_inst, cached_train_tasks, cached_ev_f, cached_ev_y,
                        bounds, RIDGE_LAMBDAS, verbose=False
                    )
                    res["p_grad"] = p_grad
                    res["eff_rank"] = float(eff_ranks_backbone[-1])
                    res["fdr_t1"] = fdr_t1
                    res["fdr_unseen"] = fdr_unseen
                    res["ram_mb"] = res.get("ram_mb", 0.0)
                    res["exp_id"] = exp_id
                    res["head"] = h_name

                    key = exp_id if len(ACTIVE_HEADS) == 1 else f"{exp_id}_{h_name}"
                    val_runs[key].append(res)
                    print(f"    [VAL  | {exp_id:<20} | {h_name:<18}] Last: {res['last_acc']:5.2f}% | Avg: {res['avg_acc']:5.2f}% | Fgt: {res['fgt']:4.2f}% | TA: {res['acc_ta']:5.2f}% | EffRank: {res['eff_rank']:5.1f} | RAM: {res['ram_mb']:5.1f}MB")

            del ev_preloaded, ev_loader, cached_train_loaders, feature_cache, trained_peft_cache
            import gc
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        print(f"\n{'='*100}\n📊 BẢNG TỔNG KẾT PHASE 1 (VAL ABLATION 80/20, 3 SEEDS): {d_name}\n{'='*100}")
        val_summary = OrderedDict()
        for key, run_list in val_runs.items():
            if not run_list:
                continue
            exp_p = run_list[0]["exp_id"]
            head_p = run_list[0]["head"]
            lasts = [r["last_acc"] for r in run_list]
            avgs  = [r["avg_acc"] for r in run_list]
            fgts  = [r["fgt"] for r in run_list]
            tas   = [r["acc_ta"] for r in run_list]
            eranks = [r["eff_rank"] for r in run_list]
            fdr1s  = [r["fdr_t1"] for r in run_list]
            fdruns = [r["fdr_unseen"] for r in run_list]
            val_summary[key] = {
                "backbone": SWIN_ABLATION_EXPERIMENTS[exp_p]["backbone"],
                "fusion":   SWIN_ABLATION_EXPERIMENTS[exp_p]["fusion"],
                "peft":     SWIN_ABLATION_EXPERIMENTS[exp_p]["peft"],
                "head":     head_p,
                "last_mean": float(np.mean(lasts)), "last_std": float(np.std(lasts)),
                "avg_mean":  float(np.mean(avgs)),  "avg_std":  float(np.std(avgs)),
                "fgt_mean":  float(np.mean(fgts)),
                "ta_mean":   float(np.mean(tas)),
                "eff_rank_mean": float(np.mean(eranks)),
                "fdr_t1_mean":   float(np.mean(fdr1s)),
                "fdr_unseen_mean": float(np.mean(fdruns)),
                "ram_mb": run_list[0].get("ram_mb", 0.0),
                "raw_runs": run_list
            }

        print(format_swin_table(val_summary))
        val_save_path = os.path.join(d_out, f"Swin_Val_Ablation_Summary_{d_name}.json")
        with open(val_save_path, "w", encoding="utf-8") as f:
            json.dump(val_summary, f, indent=2, ensure_ascii=False)
        print(f"[*] Đã lưu kết quả Val Ablation (Phase 1): {val_save_path}")

        del val_summary, val_runs
        import gc
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()


    # ==========================================================================
    # GIAI ĐOẠN 2: FINAL TEST TRÊN 100% TRAIN & OFFICIAL TEST SET
    # ==========================================================================
    if RUN_FINAL_TEST:
        print(f"\n{'='*100}\n🏆 GIAI ĐOẠN 2: FINAL OFFICIAL TEST (100% TRAIN -> OFFICIAL TEST SET, 3 SEEDS)\n{'='*100}")
        active_exps = OrderedDict((k, v) for k, v in SWIN_ABLATION_EXPERIMENTS.items() if k in ACTIVE_ABLATION_EXPERIMENTS)
        test_runs = OrderedDict()
        for exp_id in active_exps.keys():
            for h_name in ACTIVE_HEADS:
                key = exp_id if len(ACTIVE_HEADS) == 1 else f"{exp_id}_{h_name}"
                test_runs[key] = []

        N_total_train = len(raw_train[1]) if is_cifar else len(raw_train)
        N_total_test  = len(raw_test[1])  if is_cifar else len(raw_test)
        all_train_indices = np.arange(N_total_train)
        all_test_indices  = np.arange(N_total_test)
        targets_all_train = raw_train[1] if is_cifar else [y for _, y in raw_train]

        for seed in SEEDS_TO_RUN:
            print(f"\n--- [FINAL TEST - SEED {seed}] 100% Train ({N_total_train} mẫu) -> Official Test ({N_total_test} mẫu) ---")
            seed_everything(seed)
            class_order, remap, bounds, _, _ = setup_cil_seed(
                raw_train, seed, spec["n_classes"], spec["task_sizes"], is_cifar=is_cifar
            )

            ev_preloaded = precompute_eval_tensors(raw_test, all_test_indices, remap, eval_tf, is_cifar=is_cifar, desc=f"Official Test (Seed {seed})")
            ev_loader = DataLoader(ev_preloaded, batch_size=ev_bs, shuffle=False, num_workers=0, pin_memory=True)

            cached_train_loaders = {}
            for t, (lo, hi) in enumerate(bounds):
                t_train_idx = [i for i in all_train_indices if lo <= remap[int(targets_all_train[i])] < hi]
                t_preloaded = precompute_eval_tensors(raw_train, t_train_idx, remap, eval_tf, is_cifar=is_cifar, desc=f"100% Train Task {t+1}")
                cached_train_loaders[t] = (DataLoader(t_preloaded, batch_size=ev_bs, shuffle=False, num_workers=0, pin_memory=True), lo, hi)

            feature_cache = {}
            trained_peft_cache = {}

            for exp_id, exp_spec in active_exps.items():
                bb_choice  = exp_spec["backbone"]
                fs_choice  = exp_spec["fusion"]
                pf_choice  = exp_spec["peft"]
                use_flip   = exp_spec.get("flip_gram", False)
                use_sc_mix = exp_spec.get("supcon_mix", False)

                peft_train_key = (bb_choice, fs_choice, pf_choice, use_sc_mix)
                cache_key      = (bb_choice, fs_choice, pf_choice, use_sc_mix, use_flip)

                if cache_key not in feature_cache:
                    if peft_train_key not in trained_peft_cache:
                        if bb_choice == "vit_base":
                            backbone = load_pretrained_vit().to(DEVICE)
                            peft_mod = APTPrompt(depth=12, dim=768, heads=12, ema_coeff=spec.get("vit_ema", 0.70)).to(DEVICE)
                            in_feat_dim = 768
                        else:
                            backbone = SwinBackboneWrapper(arch_key=bb_choice, fusion_mode=fs_choice).to(DEVICE)
                            swin_rank  = exp_spec.get("rank", spec.get("rank", 64))
                            swin_scale = exp_spec.get("scale", spec.get("scale", 0.8))
                            swin_drop  = exp_spec.get("adapter_dropout", spec.get("adapter_dropout", 0.1))
                            peft_mod = backbone.build_peft(
                                peft_mode=pf_choice, rank=swin_rank, scale=swin_scale, dropout=swin_drop
                            )
                            in_feat_dim = backbone.out_features

                        t1_lo, t1_hi = bounds[0]
                        t1_train_idx = [i for i in all_train_indices if t1_lo <= remap[int(targets_all_train[i])] < t1_hi]
                        if peft_mod is not None:
                            t1_ds = CachedTrainSubset(raw_train, t1_train_idx, remap, train_tf, is_cifar=is_cifar)
                            p_grad, c_grad = train_first_session_peft(
                                backbone, peft_mod, t1_ds, t1_hi - t1_lo, spec, in_feat_dim, use_supcon_mix=use_sc_mix
                            )
                            del t1_ds
                        else:
                            p_grad, c_grad = 0.0, 0.0
                        trained_peft_cache[peft_train_key] = (backbone, peft_mod, in_feat_dim, p_grad)

                    backbone, peft_mod, in_feat_dim, p_grad = trained_peft_cache[peft_train_key]
                    t1_lo, t1_hi = bounds[0]

                    cached_ev_f, cached_ev_y = extract_features_loader(backbone, peft_mod, ev_loader, use_flip_gram=use_flip)
                    fdr_t1, fdr_unseen = compute_fisher_discriminant_ratios(cached_ev_f, cached_ev_y, t1_hi)

                    cached_train_tasks, eff_ranks_backbone = [], []
                    for t in range(len(bounds)):
                        t_ld, lo, hi = cached_train_loaders[t]
                        t_f, t_y = extract_features_loader(backbone, peft_mod, t_ld, use_flip_gram=use_flip)
                        cached_train_tasks.append((t_f, t_y, lo, hi))
                        mask_seen = (cached_ev_y < hi)
                        eff_ranks_backbone.append(compute_feature_eff_rank(cached_ev_f[mask_seen]))

                    feature_cache[cache_key] = (cached_train_tasks, cached_ev_f, cached_ev_y, eff_ranks_backbone, p_grad, fdr_t1, fdr_unseen, in_feat_dim)

                cached_train_tasks, cached_ev_f, cached_ev_y, eff_ranks_backbone, p_grad, fdr_t1, fdr_unseen, in_feat_dim = feature_cache[cache_key]

                for h_name in ACTIVE_HEADS:
                    # Nếu là ViT-B/16 (Exp0_Baseline, 768d), dùng cb_gamma=0.32, wa_beta=0.80 chuẩn gốc ViT; nếu là Swin, dùng từ spec:
                    cb_val = 0.32 if (bb_choice == "vit_base" and spec.get("cb_gamma", 0.0) > 0) else spec.get("cb_gamma", 0.0)
                    wa_val = 0.80 if (bb_choice == "vit_base" and spec.get("wa_beta", 0.0) > 0) else spec.get("wa_beta", 0.0)
                    head_inst = build_head(h_name, in_feat_dim, spec["n_classes"], seed=1993, cb_gamma=cb_val, wa_beta=wa_val).to(DEVICE)
                    res = train_and_eval_continual(
                        head_inst, cached_train_tasks, cached_ev_f, cached_ev_y,
                        bounds, RIDGE_LAMBDAS, verbose=False
                    )
                    res["p_grad"] = p_grad
                    res["eff_rank"] = float(eff_ranks_backbone[-1])
                    res["fdr_t1"] = fdr_t1
                    res["fdr_unseen"] = fdr_unseen
                    res["ram_mb"] = res.get("ram_mb", 0.0)
                    res["exp_id"] = exp_id
                    res["head"] = h_name

                    key = exp_id if len(ACTIVE_HEADS) == 1 else f"{exp_id}_{h_name}"
                    test_runs[key].append(res)
                    print(f"    [TEST | {exp_id:<20} | {h_name:<18}] Last: {res['last_acc']:5.2f}% | Avg: {res['avg_acc']:5.2f}% | Fgt: {res['fgt']:4.2f}% | TA: {res['acc_ta']:5.2f}% | EffRank: {res['eff_rank']:5.1f} | RAM: {res['ram_mb']:5.1f}MB")

            del ev_preloaded, ev_loader, cached_train_loaders, feature_cache, trained_peft_cache
            import gc
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        print(f"\n{'='*100}\n🏆 BẢNG TỔNG KẾT PHASE 2 (OFFICIAL TEST SET - 100% TRAIN, 3 SEEDS): {d_name}\n{'='*100}")
        test_summary = OrderedDict()
        for key, run_list in test_runs.items():
            if not run_list:
                continue
            exp_p = run_list[0]["exp_id"]
            head_p = run_list[0]["head"]
            lasts = [r["last_acc"] for r in run_list]
            avgs  = [r["avg_acc"] for r in run_list]
            fgts  = [r["fgt"] for r in run_list]
            tas   = [r["acc_ta"] for r in run_list]
            eranks = [r["eff_rank"] for r in run_list]
            fdr1s  = [r["fdr_t1"] for r in run_list]
            fdruns = [r["fdr_unseen"] for r in run_list]
            test_summary[key] = {
                "backbone": SWIN_ABLATION_EXPERIMENTS[exp_p]["backbone"],
                "fusion":   SWIN_ABLATION_EXPERIMENTS[exp_p]["fusion"],
                "peft":     SWIN_ABLATION_EXPERIMENTS[exp_p]["peft"],
                "head":     head_p,
                "last_mean": float(np.mean(lasts)), "last_std": float(np.std(lasts)),
                "avg_mean":  float(np.mean(avgs)),  "avg_std":  float(np.std(avgs)),
                "fgt_mean":  float(np.mean(fgts)),
                "ta_mean":   float(np.mean(tas)),
                "eff_rank_mean": float(np.mean(eranks)),
                "fdr_t1_mean":   float(np.mean(fdr1s)),
                "fdr_unseen_mean": float(np.mean(fdruns)),
                "ram_mb": run_list[0].get("ram_mb", 0.0),
                "raw_runs": run_list
            }

        print(format_swin_table(test_summary))
        test_save_path = os.path.join(d_out, f"Swin_Final_Test_Summary_{d_name}.json")
        with open(test_save_path, "w", encoding="utf-8") as f:
            json.dump(test_summary, f, indent=2, ensure_ascii=False)
        print(f"[*] Đã lưu kết quả Final Test (Phase 2): {test_save_path}")

        del raw_train, raw_test, train_tf, eval_tf, test_summary, test_runs
        import gc
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        zip_out = "/kaggle/working/Swin_SO_KAC_Final_Test_Outputs.zip" if IS_KAGGLE else "./results/apt/Swin_SO_KAC_Final_Test_Outputs.zip"
        if os.path.exists(MASTER_OUT_DIR):
            shutil.make_archive(zip_out.replace(".zip", ""), "zip", MASTER_OUT_DIR)
            print(f"\n📦 ĐÃ ĐÓNG GÓI ZIP KẾT QUẢ FINAL TEST TẠI: {zip_out}")
